# 05-04. 탐지 근거와 최종 확인 LOT·구간


**결론 먼저**

- **확실히 볼 것:** **240069**는 CP-Z4가 7번 벗어났다(07/07 19:00–19:41에 6번, 07/08 02:31에 +0.459). **240037**은 3번(최대 +0.157). 14 LOT 가운데 여러 번 벗어난 LOT은 이 둘뿐이다.
- **확실히 볼 것 (LOT 전체):** **240091**(SPHC)·**240121**(SPCE)은 과시효 Z7·Z8이 같은 강종 다른 LOT보다 약 3℃ 높고, 같은 강종 안에서 혼자 떨어져 있다.
- **지켜볼 것:** CP가 한 번만 벗어난 4 LOT, SPCEN 240038·240114의 Z7·Z8 차이(LOT이 2개뿐이라 어느 쪽이 다른지 판단 불가). **240108**의 짧은 변화는 세정 입구의 반복 파형으로 설명된다.

[최종 선별로 바로 가기](#final-focus)

**담당:** 정수진 · **Issue:** #65

시계열에서 관측한 구간과 통계 탐지 후보를 대조한다. 조건별 기준선·이동 Z-score·IsolationForest는 후보 탐지용이며 미래 온도 예측을 수행하지 않는다.

| 파일 | 읽는 흐름 |
|---|---|
| [01 자료·품질·시간 지도](05_all_data_timeseries_jinny_01.ipynb) | 자료 범위 → 연간 흐름 → 월별 관측창·정비 → 품질 |
| [02 LOT·강종별 연결 대조](05_all_data_timeseries_jinny_02.ipynb) | 각 LOT 전체 → 특이 구간 확대 → 이력 → 동일 강종 비교 → 해당 원본 |
| [03 전체 통계·공통 패턴](05_all_data_timeseries_jinny_03.ipynb) | 수준·IQR·시간 변화 → 구역별 응답 → CP·연관성 |
| [04 탐지·구간 근거](05_all_data_timeseries_jinny_04.ipynb) | 기준선·이동 Z·IsolationForest → 후보 위치 → 근거·해석 범위 |

대상은 CAL01의 제공된 14 LOT·22개 센서와 CAL01 정비/같은 WO의 부품교체, CR1 강종변경·정기수리다. 관측 범위는 실제 가동시간이나 소재 입·출측 시간이 아니다. 같은 강종·같은 경과 분은 비교 기준이며 같은 설정값·속도·두께를 보장하지 않는다. CP-Z4·CP-Z4M은 무차원이며, 물리적 정의와 산식은 별도로 확인해야 한다. 시간 겹침·선후관계만으로 고장 원인이나 정비 효과를 확정하지 않는다.

**이 파일에서 바로 보기:** [탐지 기준](#detector-definitions) · [탐지 위치](#detector-locations) · [근거로 좁히기](#evidence-scope) · [CP 전체](#cp-all) · [Z7·Z8 같은 강종 비교](#level-compare) · **[최종 LOT·구간](#final-focus)** · [부록: 126개 기록 연결](#interval-evidence) · [부록: 분석 범위](#analysis-scope) · [부록: 전수 목록](#all-intervals)

## 기본 설정

관측 시작·종료는 제공된 기록의 범위이며 소재의 실제 입·출측 시각이 아닙니다. TC의 ℃와 OP의 %는 기존 분석의 표시 가정을 유지합니다. 원본에 단위 사전·목표값·운전 모드가 없어 공학적 정상 한계는 정하지 않습니다. CP-Z4M은 무차원 기록값으로 원본 값을 그대로 표시합니다.

In [ ]:
from pathlib import Path
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
# 이걸 기준으로 코드 작성할 것
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/T-CR1-CAL01_온도.csv").exists() and (p / "notebooks").is_dir())
DATA_DIR = str(PROJECT_ROOT / "data")
FIG_DIR = str(PROJECT_ROOT / "figures")
os.makedirs(FIG_DIR, exist_ok=True)

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")

ENC = "utf-8-sig"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

## 자료 읽기와 연결

기존 01과 같은 규칙으로 읽습니다. LOT 시작과 CR1 강종변경 시각의 정확 일치를 확인하고, CAL01 설비의 WO와 부품교체를 실제 키로 연결합니다. 수리의 마지막 날짜는 하루 전체를 포함하는 계획으로 읽습니다.

In [ ]:
NOTEBOOK_NO = "04"
from pathlib import Path
from IPython.display import display, Markdown, Image
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from scipy.signal import find_peaks

files = {"온도": TEMP_CSV, "조업": EVENT_CSV, "수리": REPAIR_CSV,
         "설비": EQP_CSV, "부품": PART_CSV, "정비": MAINT_CSV,
         "교체": PARTCHG_CSV}
frames = {name: pd.read_csv(path, encoding=ENC) for name, path in files.items()}
temp, event, repair, eqp = [frames[k] for k in ["온도", "조업", "수리", "설비"]]
part, maint, partchg = [frames[k] for k in ["부품", "정비", "교체"]]
for frame, columns in [(temp, ["MEAS_DT"]), (event, ["EVT_DT"]),
                       (repair, ["STRT_DT", "END_DT"]),
                       (maint, ["DETC_DT", "STRT_DT", "END_DT"])]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column])
CAL_TAG = "P1-CR1-CAL01"
YEAR_START, YEAR_END = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")
grade_evt = event.loc[event.PLANT_CD.eq("CR1") & event.EVT_TYPE.eq("강종변경")].sort_values("EVT_DT")
assert not grade_evt.EVT_DT.duplicated().any()
lots = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), N_ROWS=("MEAS_DT", "size"))
lots = lots.merge(grade_evt[["EVT_DT", "AFT_VAL"]], left_on="START", right_on="EVT_DT",
                  how="left", validate="one_to_one").rename(columns={"AFT_VAL": "GRADE"})
lots = lots.sort_values("START").reset_index(drop=True)
assert lots.GRADE.notna().all()
lots["MINUTES"] = (lots.END - lots.START).dt.total_seconds() / 60
lot_meta = lots.set_index("LOT_NO")
cal_work = maint.loc[maint.EQP_TAG.eq(CAL_TAG)].sort_values("STRT_DT").copy()
assert cal_work.WO_NO.is_unique and cal_work.END_DT.ge(cal_work.STRT_DT).all()
cal_parts = (partchg.merge(cal_work[["WO_NO", "EQP_TAG", "MNT_TYPE", "DETC_DT", "STRT_DT", "END_DT", "REMARK"]],
                           on="WO_NO", validate="many_to_one")
             .merge(part, on="PART_CD", validate="many_to_one").sort_values("END_DT"))
plans = repair.loc[repair.PLANT_CD.eq("CR1")].sort_values("STRT_DT").copy()
plans["END_EXCL"] = plans.END_DT.dt.normalize() + pd.Timedelta(days=1)
TC = ["CLN-IN"] + [f"TC-Z{i}" for i in range(1, 10)] + ["TC-OUT1", "TC-OUT2"]
OP = [f"OP-Z{i}" for i in range(1, 9)]
SIGNALS = TC + OP + ["CP-Z4", "CP-Z4M"]
HEAT = [f"TC-Z{i}" for i in range(3, 7)]
PROCESS = {"CLN-IN": "세정", **{f"TC-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 10)},
           "TC-OUT1": "냉각", "TC-OUT2": "냉각",
           **{f"OP-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 9)},
           "CP-Z4": "가열·균열", "CP-Z4M": "가열·균열"}
LOT_COLOR = {240001: "#455a64", 240024: "#d55e00", 240037: "#0072b2", 240038: "#0072b2",
             240044: "#0072b2", 240058: "#a23b57", 240061: "#e69f00", 240069: "#7b2cbf",
             240091: "#009e73", 240108: "#0072b2", 240114: "#e69f00", 240117: "#7b2cbf",
             240121: "#009e73", 240133: "#7b2cbf"}
COL = {"LOT": "#087f9c", "GRADE": "#477eb5", "BM": "#b52a49",
       "CBM": "#d05c76", "TBM": "#e38592", "PART": "#7540a8", "PLAN": "#b8e2f2",
       "WORK": "#9a6324", "DETECT": "#34495e", "FOCUS": "#c83745", "QUALITY": "#bfc8cf"}
DUMMY_PRIOR = {"SPHC": "사용자 후보 가정", "SPCC": "사용자 후보 가정", "SPCE": "사용자 후보 가정",
               "SPCEN": "사용자 후보 가정", "SPFH590": "사용자 제외 가정", "DP590": "사용자 제외 가정",
               "SGCC": "사용자 제외 가정", "SPCD": "추가 검토"}
lots["DUMMY_PRIOR"] = lots.GRADE.map(DUMMY_PRIOR)
context_rows = []
for r in lots.itertuples():
    tasks = cal_work.loc[(cal_work.STRT_DT < r.END) & (cal_work.END_DT > r.START)]
    detected = cal_work.loc[cal_work.DETC_DT.between(r.START, r.END)]
    relationships = []
    for p in plans.itertuples():
        before = lots.loc[lots.END < p.STRT_DT].tail(1)
        after = lots.loc[lots.START >= p.END_EXCL].head(1)
        if len(before) and before.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 전 마지막 관측")
        if len(after) and after.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 후 첫 관측")
    context_rows.append({"LOT": int(r.LOT_NO), "GRADE": r.GRADE, "START": r.START, "END": r.END,
                         "MINUTES": r.MINUTES, "CAL_WORK": len(tasks), "CAL_DETECT": len(detected),
                         "PLAN_RELATION": " · ".join(relationships),
                         "DUMMY_PRIOR": DUMMY_PRIOR[r.GRADE]})
context = pd.DataFrame(context_rows).sort_values("START")
OUT_DIR = Path(FIG_DIR) / f"05_all_data_timeseries_jinny_{NOTEBOOK_NO}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
figure_checks = []
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)

def note(text):
    display(Markdown(text))

def table(frame, columns=None):
    shown = frame if columns is None else frame[columns]
    # 작은 CP-Z4M도 0으로 반올림하지 않고 과학 표기법으로 보여준다.
    with pd.option_context("display.float_format", lambda value: f"{value:.4g}"):
        display(shown)

# 정비 종류는 색으로도 구분한다.
COL.update({"BM":"#b52a49","CBM":"#007f86","TBM":"#825a30"})

## 그림 표시 규칙

기존 01~04의 그림 규칙을 그대로 씁니다. 짧은 제목, 공정과 컬럼이 보이는 소제목, 축의 대상·단위, 색 범례를 사용하고 TC는 왼쪽·대응 OP는 오른쪽, CP-Z4는 왼쪽·CP-Z4M은 오른쪽에 둡니다. 저장 크기에서 글자 겹침·잘림과 범례의 그래프 침범을 검사합니다.

**추가한 규칙:** 모든 그림 오른쪽 위에 시간축 태그를 붙입니다. 실제 사건·원본 확대는 `TIME AXIS · 실제 측정 날짜·시각`, 여러 LOT 형태 비교는 `TIME AXIS · 관측 시작 후 경과시간[분]`, 시간축이 없는 요약·결과표는 `COMPARISON · 비교 대상`입니다. 실제 시각 원본 확대에는 위쪽 보조축으로 같은 LOT의 경과 분을 함께 적어, 경과시간 비교 그림의 위치를 바로 찾을 수 있게 합니다.

**CP 표기:** CP-Z4와 CP-Z4M은 모두 무차원이다. 그래프 축은 각각 `CP-Z4 기록값`, `CP-Z4M 기록값`으로 표시한다. CP-Z4M의 `1e-10` 또는 `×1e-10`은 눈금 배율이며 단위가 아니다. 계측 산식과 설비·부품 연결은 단위와 별도로 확인한다.

**범례 이름:** 색·선·마커가 뜻하는 자료와 작업을 적는다. 작업 유형·내용·시간은 남기고, 띠 순서나 화면 위치는 이름에 붙이지 않는다. 확대 표시 `(a)~(c)`와 실제 시간의 전·후 구분은 유지한다.

노트북에는 코드가 그린 그림이 바로 표시되고, 같은 그림이 원본 해상도 PNG로 `figures/05_all_data_timeseries_jinny_04/<구분 폴더>/`에 저장된다. 계산 코드는 접어 두고 그림·표·해석은 펼쳐 두었다.

In [ ]:
rcParams.update({"font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
                 "xtick.labelsize": 10, "ytick.labelsize": 10.5, "savefig.dpi": 160})

def unit(signal):
    if signal in TC:
        return "온도[℃]"
    if signal in OP:
        return "제어출력[%]"
    if signal == "CP-Z4M":
        return "CP-Z4M 기록값"
    if signal == "CP-Z4":
        return "CP-Z4 기록값"
    return signal

def style(ax, xlabel="관측 시작 후 경과시간[분]"):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#a9bac7")
    ax.grid(color="#e4ebf0", linewidth=.6)
    ax.set_axisbelow(True)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.tick_params(pad=5)

def header(fig, title, subtitle="", handles=(), ncol=4):
    fig.suptitle(title, x=.035, y=.975, ha="left", va="top",
                 fontsize=20, fontweight="bold", color="#17344d")
    if subtitle:
        fig.text(.035, .924, subtitle, ha="left", va="top", fontsize=10.8, color="#526d80")
    if handles:
        fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.031, .879),
                   ncol=ncol, frameon=False, fontsize=10.5, handlelength=2.4,
                   columnspacing=1.8, labelspacing=.65)

def lot_handles(ids):
    return [Line2D([0], [0], color=LOT_COLOR[int(k)], lw=2,
                   label=f"{int(k)} · {lot_meta.loc[int(k), 'GRADE']} · {lot_meta.loc[int(k), 'START']:%m/%d}") for k in ids]

def text_box(artist, renderer):
    patch = artist.get_bbox_patch() if hasattr(artist, "get_bbox_patch") else None
    return patch.get_window_extent(renderer) if patch is not None else artist.get_window_extent(renderer)


# 설명은 글자 높이와 짧은 간격에 맞춰 모으고, 확보한 공간을 그래프에 돌려준다.
layout_reviews = []

def compact_explanations(fig):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    axes = list(fig.axes)
    visible = [ax for ax in axes if ax.axison]
    area_before = sum(ax.get_position().width * ax.get_position().height for ax in visible)
    report = {"CAPTIONS": 0, "AREA_BEFORE": area_before, "AREA_AFTER": area_before,
              "GAIN_PCT": 0.0, "LIMITS_PRESERVED": True}
    if not axes or not visible:
        return report
    positions = {ax: ax.get_position().frozen() for ax in axes}
    bottom = min(p.y0 for p in positions.values())
    top = max(p.y1 for p in positions.values())
    captions = [text for text in fig.texts
                if text is not fig._suptitle and text.get_visible() and text.get_text()
                and text.get_position()[1] < bottom - .003]
    if not captions:
        return report
    # 표·설명 상자처럼 전체 배치가 하나인 그림은 그대로 둔다.
    for artist in fig.artists:
        if hasattr(artist, "get_xy") and hasattr(artist, "get_height"):
            if artist.get_y() < bottom and artist.get_height() > .02:
                return report
    figure_height = fig.get_figheight()
    gap = 7 / 72 / figure_height
    cursor = 10 / 72 / figure_height
    caption_ids = {id(text) for text in captions}
    heights = {id(text): text_box(text, renderer).height / fig.bbox.height for text in captions}
    for text in sorted(captions, key=lambda t: t.get_position()[1]):
        text.set_position((text.get_position()[0], cursor))
        text.set_verticalalignment("bottom")
        cursor += heights[id(text)] + gap
    # 축 아래의 눈금·단위가 설명과 겹치지 않도록 실제 글자 크기를 반영한다.
    depth = 0.0
    for ax in visible:
        if positions[ax].y0 > bottom + .025:
            continue
        labels = [ax.xaxis.label] + ax.get_xticklabels()
        boxes = [text_box(t, renderer) for t in labels if t.get_visible() and t.get_text()]
        if boxes:
            depth = max(depth, bottom - min(b.y0 for b in boxes) / fig.bbox.height)
    target_bottom = max(.065, cursor + depth + gap)
    limits = [(ax.get_xlim(), ax.get_ylim()) for ax in axes]
    if target_bottom < bottom:
        scale = (top - target_bottom) / (top - bottom)
        for ax, old in positions.items():
            ax.set_position([old.x0, target_bottom + (old.y0-bottom)*scale,
                             old.width, old.height*scale])
        for text in fig.texts:
            if id(text) in caption_ids:
                continue
            x, y = text.get_position()
            if bottom <= y <= top:
                text.set_position((x, target_bottom + (y-bottom)*scale))
    # 알파벳은 축 높이와 무관하게 파형 위 8pt 위치에 놓는다.
    for ax in axes:
        height = fig.get_figheight() * ax.get_position().height
        for text in ax.texts:
            if text.get_transform() == ax.transAxes and text.get_text() in ["(a)","(b)","(c)"]:
                text.set_y(1 + 8 / 72 / height)
                text.set_verticalalignment("bottom")
        for line in ax.lines:
            if line.get_transform() != ax.transAxes:
                continue
            y = np.asarray(line.get_ydata(), dtype=float)
            if len(y) == 2 and np.all(y > 1) and np.max(y) < 1.2:
                line.set_ydata([1 + 1 / 72 / height, 1 + 7 / 72 / height])
    assert limits == [(ax.get_xlim(), ax.get_ylim()) for ax in axes], "배치 변경으로 축 범위가 달라짐"
    fig.canvas.draw()
    area_after = sum(ax.get_position().width * ax.get_position().height for ax in visible)
    report.update({"CAPTIONS": len(captions), "AREA_AFTER": area_after,
                   "GAIN_PCT": 100*(area_after/area_before-1)})
    return report

# 저장 파일과 노트북 화면 모두 그림 네 변의 여백을 같게(0.8인치) 둔다.
PAD = .8
rcParams["savefig.pad_inches"] = PAD

def finish(fig, name, svg=False):
    """저장할 크기에서 모든 글자와 범례의 겹침·잘림을 확인한다."""
    fig.set_dpi(160)
    layout = compact_explanations(fig)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for ax in fig.axes:
        for tag in ax.texts:
            if getattr(tag, "_review_label", False):
                assert not text_box(tag, renderer).overlaps(ax.get_window_extent(renderer)), "확인 표시가 파형을 가림"
    artists = list(fig.texts) + list(fig.legends)
    for ax in fig.axes:
        if not ax.axison:
            continue
        artists += [ax.title, ax._left_title, ax.xaxis.label, ax.yaxis.label,
                    ax.xaxis.offsetText, ax.yaxis.offsetText] + list(ax.texts)
        for axis in [ax.xaxis, ax.yaxis]:
            low, high = sorted(axis.get_view_interval())
            for tick in axis.get_major_ticks():
                if low <= tick.get_loc() <= high:
                    artists += [tick.label1, tick.label2]
    unique = {}
    for artist in artists:
        if not artist.get_visible():
            continue
        if hasattr(artist, "get_text") and not artist.get_text():
            continue
        unique[id(artist)] = artist
    boxes = [(artist, text_box(artist, renderer)) for artist in unique.values()]
    for artist, box in boxes:
        if box.x0 < -.5 or box.y0 < -.5 or box.x1 > fig.bbox.x1 + .5 or box.y1 > fig.bbox.y1 + .5:
            raise RuntimeError(f"글자 잘림: {name}: {getattr(artist, 'get_text', lambda: '범례')()}")
    for i, (artist, box) in enumerate(boxes):
        for other, other_box in boxes[i + 1:]:
            if box.overlaps(other_box):
                raise RuntimeError(f"글자 겹침: {name}: {getattr(artist, 'get_text', lambda: '범례')()} / {getattr(other, 'get_text', lambda: '범례')()}")
    for legend in fig.legends:
        box = legend.get_window_extent(renderer)
        if any(box.overlaps(ax.get_window_extent(renderer)) for ax in fig.axes if ax.axison):
            raise RuntimeError(f"범례가 그래프를 가림: {name}")
    path = OUT_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160, facecolor="white", bbox_inches="tight", pad_inches=PAD)
    if svg:
        vector = path.with_suffix(".svg")
        fig.savefig(vector, facecolor="white", bbox_inches="tight", pad_inches=PAD)
        vector.write_text("\n".join(line.rstrip() for line in vector.read_text(encoding="utf-8").splitlines()) + "\n", encoding="utf-8")
    figure_checks.append({"FILE": name, "TITLE": fig._suptitle.get_text(), "N_AXES": len(fig.axes),
                          "TEXT_OVERLAP": "통과", "TEXT_BOUNDS": "통과", "LEGEND_BOUNDS": "통과"})
    layout_reviews.append({"FILE": name, **layout})
    # 저장한 파일을 다시 불러오지 않고, 방금 그린 그림을 노트북에 바로 그린다.
    # 파일은 위에서 원본 해상도(160dpi)로 저장했고, 노트북 화면용으로만 해상도를 낮춘다.
    fig.set_dpi(72)
    display(fig)
    plt.close(fig)

In [ ]:
import matplotlib

# 시간축 태그: 그림 오른쪽 위에 이 그림을 어떤 기준으로 읽는지 적는다.
TAG_REAL = "TIME AXIS · 실제 측정 날짜·시각"
TAG_ELAPSED = "TIME AXIS · 관측 시작 후 경과시간[분]"
TAG_REL = "TIME AXIS · LOT 시작 기준 상대시간[시간]"

def axis_tag(fig, text):
    fig.text(.975, .978, text, ha="right", va="top", fontsize=13, color="#17344d",
             bbox=dict(boxstyle="round,pad=.45", fc="#eef3f7", ec="#9fb3c3", lw=.9))

def compare_tag(target):
    return f"COMPARISON · {target}"

def elapsed_top(ax, start, label="경과[분]"):
    """실제 시각 축 위에 같은 LOT의 관측 시작 후 경과 분을 함께 적는다."""
    base = mdates.date2num(start)
    top = ax.secondary_xaxis("top", functions=(lambda x: (x - base) * 1440, lambda m: base + m / 1440))
    top.set_xlabel(label, fontsize=11, color="#526d80", labelpad=6)
    top.tick_params(labelsize=10, colors="#526d80")
    top.xaxis.set_major_locator(MaxNLocator(6))
    return top

def compact_header(fig, sub_y, legend_y, title_size=25, sub_size=14, legend_size=13):
    """키 큰 그림에서 제목·부제·범례 사이 빈 공간을 줄이고 글자를 키운다."""
    fig._suptitle.set_fontsize(title_size)
    for text in fig.texts:
        if np.isclose(text.get_position()[1], .924):
            text.set_position((.035, sub_y))
            text.set_fontsize(sub_size)
    if fig.legends:
        fig.legends[0].set_bbox_to_anchor((.031, legend_y))
        for text in fig.legends[0].get_texts():
            text.set_fontsize(legend_size)

def shades(color, n=4):
    """LOT 색 하나를 진한 색 → 밝은 색으로 나눈다. 구역 순서를 같은 LOT 안에서 구분한다."""
    rgb = np.array(matplotlib.colors.to_rgb(color))
    dark, light = rgb * .55, rgb * .62 + .38
    return [tuple(dark * (1 - t) + light * t) for t in np.linspace(0, 1, n)]

## 상세 시계열 준비

1초 기대 간격의 품질, 1분 집계, 사후 형태 상태(초기 승온·종료 전환·안정 참고), 같은 강종·안정 참고·같은 경과 분 비교, 실제 작업 교집합을 계산한다. 기존 01~08과 같은 코드다.

In [ ]:
# 1초 기대 간격은 원본 시각 차이의 최빈값으로 확인한다. 결측은 보간하지 않는다.
raw, profile, quality, expected, states = {}, {}, {}, {}, {}
quality_rows, state_rows = [], []
STATE_COLOR = {"안정 참고": "#009e73", "초기 승온": "#e69f00", "종료 전환": "#d55e00"}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = temp.loc[temp.LOT_NO.eq(k)].sort_values("MEAS_DT").copy()
    assert not g.MEAS_DT.duplicated().any()
    gaps_s = g.MEAS_DT.diff().dt.total_seconds().dropna()
    assert gaps_s.mode().iloc[0] == 1 and gaps_s.ge(1).all()
    g["ELAPSED"] = (g.MEAS_DT - r.START).dt.total_seconds() / 60
    g["MINUTE"] = np.floor(g.ELAPSED).astype(int)
    grid = pd.RangeIndex(int(np.floor(r.MINUTES)) + 1, name="MINUTE")
    p = g.groupby("MINUTE")[SIGNALS].agg(["median", "min", "max", "count"]).reindex(grid)
    exp = pd.Series(np.minimum(60, int((r.END-r.START).total_seconds()) + 1 - grid.to_numpy()*60), index=grid)
    counts = g.groupby("MINUTE").size().reindex(grid, fill_value=0)
    q = p.xs("count", axis=1, level=1).fillna(0).div(exp, axis=0)
    raw[k], profile[k], quality[k], expected[k] = g, p, q, exp
    for s in SIGNALS:
        quality_rows.append({"LOT": k, "GRADE": r.GRADE, "SENSOR": s, "ROWS": len(g),
                             "EXPECTED_SECONDS": int(exp.sum()), "MISSING_TIMES_S": int(exp.sum()-len(g)),
                             "SENSOR_NAN": int(g[s].isna().sum()), "VALID_RATIO": g[s].count()/exp.sum(),
                             "LOW_90_WINDOWS": int(q[s].lt(.9).sum())})
    # 사후 형태 분류다. 실제 운전 모드나 온라인 모델 입력으로 사용하지 않는다.
    st = pd.Series("안정 참고", index=grid, dtype="object")
    early = g.loc[g.ELAPSED.lt(5), HEAT].median()
    later = g.loc[g.ELAPSED.between(80,100), HEAT].median()
    rise = later - early
    warm_end, tail_start = np.nan, np.nan
    if rise.notna().all() and rise.min() >= 20 and g.ELAPSED.between(80,100).shape[0] >= 600:
        near = pd.concat([(p[(s,"median")]-later[s]).abs().le(5 if s=="TC-Z3" else 2) for s in HEAT], axis=1).all(axis=1)
        slow = pd.concat([p[(s,"median")].diff(5).abs().le(2) for s in HEAT[1:]], axis=1).all(axis=1)
        run = (near & slow & grid.to_series().ge(5)).rolling(5, min_periods=5).sum().eq(5)
        warm_end = int(run.index[run][0])+1 if run.any() else len(grid)
        st.loc[grid < warm_end] = "초기 승온"
    middle = g.loc[g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6), HEAT].median()
    tail = g.loc[g.ELAPSED.ge(max(0,r.MINUTES-5)), HEAT].median()
    if (tail-middle).notna().all() and (tail-middle).max() <= -10:
        midmask = g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6)
        drop = pd.concat([(p[(s,"median")]-middle[s]).le(-10) for s in HEAT], axis=1).sum(axis=1).ge(3)
        cp_drop = (p[("CP-Z4","median")]-g.loc[midmask,"CP-Z4"].median()).le(-.05).rolling(3).sum().eq(3)
        op = ["OP-Z4","OP-Z5","OP-Z6"]
        op_drop = (p.loc[:,[(s,"median") for s in op]].median(axis=1)-g.loc[midmask,op].median().median()).le(-10).rolling(3).sum().eq(3)
        hit = (drop | cp_drop | op_drop) & grid.to_series().ge(r.MINUTES*.6)
        if hit.any():
            tail_start = int(hit.index[hit][0])
            st.loc[grid >= tail_start] = "종료 전환"
    states[k] = st
    state_rows.append({"LOT": k, "GRADE": r.GRADE, "RISE_MIN_C": rise.min(), "RISE_MAX_C": rise.max(),
                       "WARM_END_MIN": warm_end, "TAIL_START_MIN": tail_start,
                       "STABLE_WINDOWS": int(st.eq("안정 참고").sum())})
quality_summary = pd.DataFrame(quality_rows)
state_summary = pd.DataFrame(state_rows)
def runs(mask, minimum=1):
    """연속 True의 [시작, 끝 제외) 경과 분을 돌려준다."""
    a = np.flatnonzero(mask.to_numpy(bool))
    if not len(a):
        return []
    groups = np.split(a, np.flatnonzero(np.diff(a)>1)+1)
    return [(int(mask.index[z[0]]), int(mask.index[z[-1]])+1) for z in groups if len(z)>=minimum]

In [ ]:
def line(ax, lot, signal, start=None, end=None, clock=False, raw_values=False, envelope=False):
    k = int(lot)
    if raw_values:
        r = lot_meta.loc[k]
        a = max(0, start or 0)
        b = min(r.MINUTES, r.MINUTES if end is None else end)
        # 표시용 1초 격자에만 재배치하여 결측 시각에서 선을 끊는다.
        times = pd.date_range(r.START+pd.Timedelta(seconds=int(a*60)), r.START+pd.Timedelta(seconds=int(b*60)), freq="s")
        p = raw[k].set_index("MEAS_DT").reindex(times)
        x = times if clock else (times-r.START).total_seconds()/60
        y = p[signal].to_numpy()
        low = quality[k][signal].loc[quality[k].index.to_series().between(np.ceil(a),np.floor(b))].lt(.9)
        if low.any():
            marks = low.index[low].to_numpy()
            qx = r.START + pd.to_timedelta(marks,unit="min") if clock else marks
            ax.scatter(qx,np.full(len(marks),.02),transform=ax.get_xaxis_transform(),marker="|",color="#7c8995",s=25,lw=.6,zorder=5)
    else:
        p = profile[k]
        p = p.loc[(p.index >= (0 if start is None else start)) & (p.index <= (np.inf if end is None else end))]
        x = lot_meta.loc[k,"START"] + pd.to_timedelta(p.index,unit="min") if clock else p.index.to_numpy()
        y = p[(signal,"median")].to_numpy()
        if envelope:
            ax.fill_between(x,p[(signal,"min")].to_numpy(),p[(signal,"max")].to_numpy(),color=LOT_COLOR[k],alpha=.14,lw=0)
        low = quality[k].loc[p.index,signal].lt(.9)
        if low.any():
            ax.scatter(np.asarray(x)[low],np.full(low.sum(),.02),transform=ax.get_xaxis_transform(),marker="|",color="#7c8995",s=25,lw=.6,zorder=5)
    ax.plot(x,y,color=LOT_COLOR[k],lw=1.8)
    ax.set_title(f"{PROCESS[signal]} [{signal}]",loc="left",pad=12)
    ax.set_ylabel(unit(signal),labelpad=10)
    style(ax,"관측 시각[월/일 시:분]" if clock else "관측 시작 후 경과시간[분]")
    ax.xaxis.set_major_locator(MaxNLocator(5)) if not clock else None
    ax.yaxis.set_major_locator(MaxNLocator(4))
    if signal == "CP-Z4M":
        ax.ticklabel_format(axis="y",style="sci",scilimits=(0,0))
    if clock:
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))

QUALITY_HANDLE = Line2D([0],[0],color="#7c8995",marker="|",linestyle="None",markersize=10,label="분당 유효 기록 90% 미만")
FOCUS_HANDLE = Patch(facecolor=COL["FOCUS"],alpha=.12,label="검토·확대 시간창")
ENVELOPE_HANDLE = Patch(facecolor="#97b5cc",alpha=.35,label="각 LOT 색 음영: 분당 최소–최대")

In [ ]:
# 같은 강종·안정 참고 상태·같은 경과 분에서 대상 LOT을 뺀 대조만 쓴다.
# 비교할 수 없는 후반 구간을 잘라내거나 대조 중앙값으로 채우지 않는다.
comparison, sensor_rows, episode_rows = {}, [], []
NONCYCLIC = [s for s in TC if s not in ["TC-Z9","TC-OUT1","TC-OUT2"]]
for grade, sub in lots.groupby("GRADE",sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    grid = pd.RangeIndex(max(len(profile[k]) for k in ids))
    for s in SIGNALS:
        values = pd.DataFrame({k:profile[k][(s,"median")].where(quality[k][s].ge(.9)&states[k].eq("안정 참고")).reindex(grid) for k in ids})
        for k in ids:
            peers = values.drop(columns=k)
            n = peers.count(axis=1)
            center = peers.median(axis=1)
            delta = (values[k]-center).where(n.ge(1))
            d = pd.DataFrame({"VALUE":values[k],"CENTER":center,"N_PEERS":n,"DELTA":delta})
            comparison[k,s] = d
            v = delta.dropna()
            sensor_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"COMPARE_WINDOWS":len(v),
                                "MEDIAN_DELTA":v.median(),"Q05_DELTA":v.quantile(.05),"Q95_DELTA":v.quantile(.95),
                                "MAX_ABS_DELTA":v.abs().max(),"MIN_PEERS":int(n.loc[v.index].min()) if len(v) else 0})
            if s in NONCYCLIC:
                for a,b in runs(delta.abs().ge(3)&n.ge(2),3):
                    block = d.loc[a:b-1]
                    episode_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"START_MIN":a,"END_MIN":b,
                                         "DURATION_MIN":b-a,"MAX_ABS_DELTA_C":block.DELTA.abs().max(),
                                         "MIN_PEERS":int(block.N_PEERS.min())})
sensor_summary = pd.DataFrame(sensor_rows)
episodes = pd.DataFrame(episode_rows,columns=["LOT","GRADE","SENSOR","START_MIN","END_MIN","DURATION_MIN","MAX_ABS_DELTA_C","MIN_PEERS"])
# CP는 무차원이다. 자체 참고 중앙값과 차이를 탐색하며 정상 한계로 해석하지 않는다.
cp_rows, cp_flags = [], {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = raw[k]
    ref = profile[k][("CP-Z4","median")].where(states[k].eq("안정 참고")&quality[k]["CP-Z4"].ge(.9)).median()
    hit = (g["CP-Z4"]-ref).abs().ge(.05)
    cp_flags[k] = pd.Series(profile[k].index.isin(g.loc[hit,"MINUTE"]),index=profile[k].index)
    if hit.any():
        j = (g["CP-Z4"]-ref).abs().idxmax()
        peak = g.loc[j]
        # 5초 초과 시각 공백에서 분리한다. 짧은 공백도 보간하지 않고 실제 기록 행수·최대 간격을 따로 남긴다.
        cluster = ((hit != hit.shift())|g.MEAS_DT.diff().dt.total_seconds().gt(5)).cumsum()
        pulse = g.loc[cluster.eq(cluster.loc[j])&hit]
        cp_rows.append({"LOT":k,"GRADE":r.GRADE,"BASE":ref,"PEAK_MIN":peak.ELAPSED,"PEAK_TIME":peak.MEAS_DT,
                        "PEAK_VALUE":peak["CP-Z4"],"DELTA":peak["CP-Z4"]-ref,
                        "START_MIN":pulse.ELAPSED.min(),"END_MIN":pulse.ELAPSED.max()+1/60,
                        "PULSE_WIDTH_MIN":(pulse.MEAS_DT.max()-pulse.MEAS_DT.min()).total_seconds()/60+1/60,
                        "MAX_RECORD_GAP_S":pulse.MEAS_DT.diff().dt.total_seconds().max(),
                        "RECORDED_SECONDS":len(pulse),"TOTAL_HIT_SECONDS":int(hit.sum()),
                        "TOTAL_HIT_MIN_WINDOWS":int(cp_flags[k].sum())})
cp_candidates = pd.DataFrame(cp_rows,columns=["LOT","GRADE","BASE","PEAK_MIN","PEAK_TIME","PEAK_VALUE","DELTA","START_MIN","END_MIN","PULSE_WIDTH_MIN","MAX_RECORD_GAP_S","RECORDED_SECONDS","TOTAL_HIT_SECONDS","TOTAL_HIT_MIN_WINDOWS"])

In [ ]:
# 관측 범위와 실제 CAL01 작업의 교집합을 경과 분으로 계산한다. 같은 시각의 작업은 한 번만 센다.
def merge_ranges(intervals):
    merged = []
    for start, end in sorted(intervals):
        if end <= start:
            continue
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
        else:
            merged.append((start, end))
    return merged

work_ranges = {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    tasks = cal_work.loc[cal_work.STRT_DT.lt(r.END) & cal_work.END_DT.gt(r.START)]
    work_ranges[k] = merge_ranges([((max(r.START, t.STRT_DT) - r.START).total_seconds() / 60,
                                    (min(r.END, t.END_DT) - r.START).total_seconds() / 60) for t in tasks.itertuples()])
assert np.isclose(sum(b - a for a, b in work_ranges[240117]), 304)

In [ ]:
hypotheses = []
def verdict(hypothesis, result, evidence, reason, use, limit="운전 설정·인과·실제 고장 정보 부족", source="기존 03"):
    hypotheses.append({"가설": hypothesis, "판정": result, "검증 근거": evidence, "판정 이유": reason,
                       "08 활용": use, "남은 한계": limit, "출처": source})

def raw_window(lot, a, b):
    """1초 격자에 원본을 놓아 기록 공백에서 선을 끊는다. 값을 채우지 않는다."""
    r = lot_meta.loc[lot]
    times = pd.date_range(r.START + pd.Timedelta(seconds=int(a * 60)), min(r.END, r.START + pd.Timedelta(seconds=int(b * 60))), freq="s")
    return raw[lot].set_index("MEAS_DT").reindex(times)

def clock_axis(ax, day):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    style(ax, f"실제 측정 시각[{day:%m/%d} 시:분]")

def clock_top(ax, start, label="같은 LOT 실제 시각[시:분]"):
    """경과 분 축 위에 대상 LOT의 실제 시각을 함께 적는다."""
    base = mdates.date2num(start)
    top = ax.secondary_xaxis("top", functions=(lambda m: base + m / 1440, lambda d: (d - base) * 1440))
    top.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    top.set_xlabel(label, fontsize=11, color="#526d80", labelpad=6)
    top.tick_params(labelsize=10, colors="#526d80")
    return top

def mean30(k, cols):
    """팀 자료와 같은 집계: 센서별 유효값 30개 이상인 1분 평균."""
    g = raw[k].set_index("MEAS_DT")[cols]
    return g.resample("1min").mean().where(g.resample("1min").count().ge(30))

def sustained_from(series, limit):
    """기준보다 낮은 상태가 관측 끝까지 이어지는 구간의 시작 시각."""
    s = series.dropna(); low = s.lt(limit)
    if not low.iloc[-1]:
        return None
    i = len(low) - 1
    while i > 0 and low.iloc[i - 1]:
        i -= 1
    return low.index[i]

In [ ]:
# ------------------------------------------------------------------------------------------------
# 예정된 변화와 확실히 볼 후보 ①~④ (01·02·03 노트북이 같은 코드로 계산한다)
# ------------------------------------------------------------------------------------------------
# 예정된 변화: 계획수리 직전의 종료 전환(끄면 출력·온도가 함께 내려감), 직후의 초기 승온(켜면 함께 올라감)
relation = context.set_index("LOT").PLAN_RELATION
expected_rows = []
for k in lots.LOT_NO.astype(int):
    for state, kind in [("종료 전환", "예정된 정지"), ("초기 승온", "재가동 승온")]:
        for a, b in runs(states[k].eq(state)):
            near_plan = bool(relation.get(k))
            expected_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "종류": kind if near_plan else state + " · 일정 없음",
                                  "START_MIN": float(a), "END_MIN": float(min(b, lot_meta.loc[k, "MINUTES"])),
                                  "시작": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(a)),
                                  "끝": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(min(b, lot_meta.loc[k, "MINUTES"]))),
                                  "수리 관계": relation.get(k) or "-"})
expected = pd.DataFrame(expected_rows)

def in_expected(k, minute):
    d = expected.loc[expected.LOT.eq(k)]
    return bool(((d.START_MIN <= minute) & (d.END_MIN > minute)).any())

# ① 출력과 온도가 반대로 움직인 구간: 출력 0%(≤2%)가 5분 이상인데 온도가 내려가지 않음 / 출력 100%(≥98%)가 5분 이상인데 온도가 오르지 않음
opposite_rows = []
for k in lots.LOT_NO.astype(int):
    p = profile[k]
    for z in range(1, 9):
        op = p[(f"OP-Z{z}", "median")].where(quality[k][f"OP-Z{z}"].ge(.9))
        tc = p[(f"TC-Z{z}", "median")].where(quality[k][f"TC-Z{z}"].ge(.9))
        for kind, on, bad in [("출력 0%인데 온도 안 내려감", op.le(2), lambda d: d >= 0),
                              ("출력 100%인데 온도 안 오름", op.ge(98), lambda d: d <= 0)]:
            for a, b in runs(on.fillna(False), 5):
                change = tc.iloc[min(b, len(tc) - 1)] - tc.iloc[a]
                if bad(change):
                    opposite_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "ZONE": f"Z{z}", "종류": kind,
                                          "START_MIN": a, "END_MIN": b, "길이[분]": b - a, "온도 변화[℃]": change,
                                          "시작": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=a),
                                          "예정된 변화 중": in_expected(k, a)})
opposite = pd.DataFrame(opposite_rows)

# ②·③ 같은 강종 대비 LOT 전체 수준 차이: 안정 참고 1분 중앙값의 LOT 중앙값 - 같은 강종 다른 LOT 중앙값들의 중앙값
level_rows = []
for grade, sub in lots.groupby("GRADE", sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    for s in SIGNALS:
        med = {k: profile[k][(s, "median")].where(states[k].eq("안정 참고") & quality[k][s].ge(.9)).median() for k in ids}
        for k in ids:
            peers = [med[o] for o in ids if o != k and pd.notna(med[o])]
            level_rows.append({"LOT": k, "GRADE": grade, "SENSOR": s, "LOT_MEDIAN": med[k],
                               "PEER_MEDIAN": np.median(peers) if peers else np.nan, "N_PEERS": len(peers)})
level_gap = pd.DataFrame(level_rows)
level_gap["GAP"] = level_gap.LOT_MEDIAN - level_gap.PEER_MEDIAN
overage = level_gap.loc[level_gap.SENSOR.isin(["TC-Z7", "TC-Z8"]) & level_gap.GAP.ge(2.5)]       # ② 과시효 Z7·Z8이 같은 강종보다 2.5℃ 이상 높은 LOT
OVERAGE_LOTS = sorted(overage.LOT.unique())

# ③ 세정 입구 온도(CLN-IN) LOT 중앙값을 정렬해 가장 큰 간격에서 두 묶음으로 나눈다.
cln = level_gap.loc[level_gap.SENSOR.eq("CLN-IN")].set_index("LOT").LOT_MEDIAN.sort_values()
cut = cln.diff().idxmax()                                   # 큰 묶음의 첫 LOT
CLN_SPLIT = (cln.loc[:cut].iloc[-2] + cln.loc[cut]) / 2
cln_group = pd.Series(np.where(cln.ge(CLN_SPLIT), f"{CLN_SPLIT:.1f}℃ 이상", f"{CLN_SPLIT:.1f}℃ 미만"), index=cln.index)

# ④ CP 변화: 자체 안정 참고 기준과 가장 크게 벗어난 펄스 (예정된 정지 중의 CP 하강은 제외)
cp_main = cp_candidates.loc[[not in_expected(int(r.LOT), r.PEAK_MIN) for r in cp_candidates.itertuples()]]
CP_TOP = cp_main.loc[cp_main.DELTA.abs().idxmax()]

# ① 대표 사례: 출력 0%가 5분 이상인데 온도가 가장 크게 오른 곳
z0 = opposite.loc[opposite["종류"].str.startswith("출력 0%") & ~opposite["예정된 변화 중"]]
Z3_TOP = z0.loc[z0["온도 변화[℃]"].idxmax()]

# 03에서 자세히 본 LOT과 이유 (그림에서 빨간색으로 표시한다)
SELECTED = {}
for k in OVERAGE_LOTS:
    SELECTED.setdefault(int(k), []).append("② Z7·Z8")
SELECTED.setdefault(int(CP_TOP.LOT), []).append("④ CP")
SELECTED.setdefault(int(Z3_TOP.LOT), []).append("① Z3")
SELECT_COLOR = "#d62728"

MARK = {"①": "#c0392b", "②": "#8e44ad", "③": "#16a085", "④": "#d35400"}
MARK_NAME = {"①": "출력·온도 반대 움직임", "②": "과시효 Z7·Z8 수준 차이", "③": "세정 입구 온도 두 수준", "④": "가장 큰 CP 변화"}
EXPECTED_COLOR = "#8d99a6"

In [ ]:
from matplotlib.patches import Rectangle
GRADE_COLOR={"SGCC":"#455a64","SPFH590":"#d55e00","SPCE":"#0072b2","SPCEN":"#e69f00","SPHC":"#009e73","DP590":"#a23b57","SPCD":"#7b2cbf","SPCC":"#8c6d31"}
ZONE_COLOR={1:"#455a64",2:"#d55e00",3:"#0072b2",4:"#e69f00",5:"#009e73",6:"#cc79a7",7:"#8c6d31",8:"#7b2cbf",9:"#56b4e9"}
ZONES=[3,4,5,6]
SELECT_COLOR="#c83745"
MARK_NAME["①"]="출력·온도 응답 확인"
PROCESS["TC-Z9"]="Z9"
def sensor_color(s):
    if s.startswith(("TC-Z","OP-Z")):return ZONE_COLOR[int(s.rsplit('Z',1)[1])]
    return {"CLN-IN":"#455a64","CP-Z4":"#b04482","CP-Z4M":"#7b2cbf","TC-OUT1":"#009e73","TC-OUT2":"#d55e00"}[s]
def sensor_scale(s):return 1e10 if s=="CP-Z4M" else 1
def shown_unit(s):return "CP-Z4M 기록값[×1e-10]" if s=="CP-Z4M" else unit(s)
def when(k,a,b):
    start=lot_meta.loc[k,"START"]
    return f"{start+pd.Timedelta(minutes=float(a)):%m/%d %H:%M:%S}–{start+pd.Timedelta(minutes=float(b)):%m/%d %H:%M:%S}"
def clock_axis(ax,day=None):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
    style(ax,"실제 측정 날짜·시각[월/일 시:분]")
def raw_window(lot,a,b):
    r=lot_meta.loc[lot]
    times=pd.date_range(r.START+pd.Timedelta(seconds=int(a*60)),min(r.END,r.START+pd.Timedelta(seconds=int(b*60))),freq="s")
    return raw[lot].set_index("MEAS_DT").reindex(times)
def remark_text(t):return str(t.REMARK).strip() if pd.notna(t.REMARK) else "작업 내용 미기재"
def parts_for(wo):
    return [f"{p.PART_NM} [{p.PART_CD}] ×{p.QTY:g} ({p.CHG_RSN})" for p in cal_parts.loc[cal_parts.WO_NO.eq(wo)].itertuples()]
def linked_work(k,pad_hours=0):
    r=lot_meta.loc[k]
    return cal_work.loc[cal_work.STRT_DT.lt(r.END+pd.Timedelta(hours=pad_hours))&cal_work.END_DT.gt(r.START-pd.Timedelta(hours=pad_hours))].sort_values("STRT_DT")

# 모든 후보를 목록에 남기고, 긴 수준 차이와 짧은 변화를 구분해 표시한다.
review_rows=[]
for e in episodes.itertuples():
    level=level_gap.loc[level_gap.LOT.eq(e.LOT)&level_gap.SENSOR.eq(e.SENSOR),"GAP"].iloc[0]
    long=bool(e.DURATION_MIN>=30)
    review_rows.append({"LOT":int(e.LOT),"GRADE":e.GRADE,"SENSOR":e.SENSOR,"KIND":"같은 강종 차이","A":float(e.START_MIN),"B":float(e.END_MIN),"DETAIL":f"최대 {e.MAX_ABS_DELTA_C:.2f}℃ · 대조 {e.MIN_PEERS} LOT 이상","LOCAL":not long})
for e in opposite.itertuples():
    review_rows.append({"LOT":int(e.LOT),"GRADE":e.GRADE,"SENSOR":f"TC-{e.ZONE}","KIND":"출력·온도 응답","A":float(e.START_MIN),"B":float(e.END_MIN),"DETAIL":f"{e.종류} · TC 변화 {e._8:+.2f}℃","LOCAL":True})
for e in cp_candidates.itertuples():
    review_rows.append({"LOT":int(e.LOT),"GRADE":e.GRADE,"SENSOR":"CP-Z4","KIND":"CP 변화","A":float(e.START_MIN),"B":float(e.END_MIN),"DETAIL":f"자체 참고 대비 {e.DELTA:+.3f} · 극값 {e.PEAK_VALUE:.4g}","LOCAL":e.PULSE_WIDTH_MIN<30})
review_catalogue=pd.DataFrame(review_rows).sort_values(["LOT","A","SENSOR"]).reset_index(drop=True)
review_catalogue["REVIEW_ID"]=[f"R{i+1:03d}" for i in range(len(review_catalogue))]
review_catalogue["START"]=review_catalogue.LOT.map(lot_meta.START)+pd.to_timedelta(review_catalogue.A,unit="min")
review_catalogue["END"]=review_catalogue.LOT.map(lot_meta.START)+pd.to_timedelta(review_catalogue.B,unit="min")

def lot_windows(k):
    """겹친 국소 변화의 문맥을 합친다. 긴 수준 차이는 전체 파형·비교 표에서 확인한다."""
    rows=review_catalogue.loc[review_catalogue.LOT.eq(k)&review_catalogue.LOCAL]
    result=[]
    for r in rows.sort_values('A').itertuples():
        a=max(0,r.A-5);b=min(lot_meta.loc[k,"MINUTES"],r.B+5)
        if result and a<=result[-1]["B"]+3:
            result[-1]["B"]=max(result[-1]["B"],b);result[-1]["IDS"].append(r.REVIEW_ID);result[-1]["SENSORS"].add(r.SENSOR)
        else:result.append({"A":a,"B":b,"IDS":[r.REVIEW_ID],"SENSORS":{r.SENSOR}})
    for i,r in enumerate(result):r["WINDOW_ID"]=f"{k}-W{i+1:02d}"
    return result
overview_coverage={};window_coverage={};maintenance_stage_rows=[];window_audit=[]

<a id="detector-definitions"></a>

## 1. 탐지에 사용하는 통계와 기준

조건별 범위 기준선은 같은 강종·안정 참고·대조 2 LOT 이상에서 ±3℃·3분 지속을 확인한다. 이동 Z는 현재를 제외한 이전 30분(최소 20분) 중앙값·표준편차를 사용하며 표준편차 하한을 둔다. MAD 기반 Robust Z와는 다른 계산이다. IsolationForest는 과거 편차·변화량·변동을 사용하고 시간순 앞 8 LOT으로만 기준을 학습한다. 정답 라벨이 없어 정확도 대신 위치·일치·근거를 비교한다.

In [ ]:
# 탐지용 1분 통계를 만든다. 온도 예측 목표나 정답 라벨을 만들지 않는다.
# 작업 REMARK·교체·사후 상태·후보 플래그·전체 LOT 통계는 온라인 특징에 넣지 않는다.
def features_for_lot(g,lot,grade,start):
    g=g.sort_values("MEAS_DT").copy()
    g["MINUTE"]=np.floor((g.MEAS_DT-start).dt.total_seconds()/60).astype(int)
    grid=pd.RangeIndex(g.MINUTE.max()+1,name="MINUTE")
    grouped=g.groupby("MINUTE")[SIGNALS]
    agg=grouped.agg(["median","min","max","count"]).reindex(grid)
    iqr=(grouped.quantile(.75)-grouped.quantile(.25)).reindex(grid)
    result={"LOT":np.full(len(grid),lot),"GRADE":np.full(len(grid),grade),"MINUTE":grid.to_numpy(),"WINDOW_END_TIME":start+pd.to_timedelta(grid+1,unit="min")}
    last=g.groupby("MINUTE").MEAS_DT.max().reindex(grid)
    result["MAX_SOURCE_TIME"]=last.to_numpy()
    result["GRADE_EVENT_TIME"]=start
    for s in SIGNALS:
        # 온라인에서는 닫힌 60초 창을 기다린다. 마지막 짧은 관측도 60초를 분모로 사용한다.
        q=agg[(s,"count")].fillna(0)/60
        y=agg[(s,"median")].where(q.ge(.9))
        result[s+"_CURRENT"]=y.to_numpy()
        result[s+"_IQR"]=iqr[s].where(q.ge(.9)).to_numpy()
        result[s+"_VALID_RATIO_60"]=q.to_numpy()
        result[s+"_DELTA1"]=y.diff().to_numpy()
        past=y.shift(1)
        result[s+"_PAST5_MEDIAN"]=past.rolling(5,min_periods=3).median().to_numpy()
        result[s+"_PAST5_STD"]=past.rolling(5,min_periods=3).std().to_numpy()
        result[s+"_PAST60_Q95_Q05"]=(past.rolling(60,min_periods=30).quantile(.95)-past.rolling(60,min_periods=30).quantile(.05)).to_numpy()
        if s in ["TC-Z9","TC-OUT1","TC-OUT2"]:
            result[s+"_PAST300_Q95_Q05"]=(past.rolling(300,min_periods=180).quantile(.95)-past.rolling(300,min_periods=180).quantile(.05)).to_numpy()
    return pd.DataFrame(result)

feature_frames=[features_for_lot(raw[int(r.LOT_NO)],int(r.LOT_NO),r.GRADE,r.START) for r in lots.itertuples()]
features=pd.concat(feature_frames,ignore_index=True).sort_values(["WINDOW_END_TIME","LOT"]).reset_index(drop=True)
feature_columns=[c for c in features.columns if c not in ["LOT","GRADE","MINUTE","WINDOW_END_TIME","MAX_SOURCE_TIME","GRADE_EVENT_TIME"]]
assert features.MAX_SOURCE_TIME.dropna().lt(features.loc[features.MAX_SOURCE_TIME.notna(),"WINDOW_END_TIME"]).all()
assert features.GRADE_EVENT_TIME.le(features.WINDOW_END_TIME).all()

In [ ]:
# 4단계: 세 가지 탐지를 같은 1분 단위로 맞춘다.
from sklearn.ensemble import IsolationForest
Z_WINDOW, Z_MIN, Z_LIMIT = 30, 20, 4                  # 이전 30분(최소 20분)의 중앙값·표준편차, |Z| ≥ 4
SD_FLOOR = {"TC": .5, "OP": 2.0}                      # 아주 작은 표준편차로 Z가 커지지 않게 하는 최소값
IF_SIGNALS = NONCYCLIC + [f"OP-Z{z}" for z in range(1, 9)]
train_ids = set(lots.sort_values("START").LOT_NO.astype(int).head(8))   # 08과 같은 시간 순서 분할

def change_features(g):
    """한 LOT의 1분 특징에서 이전 30분 대비 편차·Z와 변화량 특징을 만든다. 현재 분은 기준에서 뺀다."""
    g = g.sort_values("MINUTE").set_index("MINUTE")
    out = pd.DataFrame({"LOT": g.LOT.iloc[0]}, index=g.index)
    for s in IF_SIGNALS:
        current = g[s + "_CURRENT"]
        past = current.shift(1)
        center = past.rolling(Z_WINDOW, min_periods=Z_MIN).median()
        spread = past.rolling(Z_WINDOW, min_periods=Z_MIN).std().clip(lower=SD_FLOOR["TC" if s in TC else "OP"])
        out[s + "_DEV30"] = current - center
        out[s + "_Z30"] = (current - center) / spread
        out[s + "_DELTA1"] = g[s + "_DELTA1"]
        out[s + "_PAST5_STD"] = g[s + "_PAST5_STD"]
    return out.rename_axis("MINUTE").reset_index()

detect = pd.concat([change_features(g) for _, g in features.groupby("LOT")], ignore_index=True)

# (1) 조건별 범위 기준선: 같은 강종·안정 참고·같은 경과 분 대조 중앙값과 |차이| ≥ 3℃, 대조 2 LOT 이상, 3분 연속
detect["RULE_FLAG"] = False
for e in episodes.itertuples():
    detect.loc[detect.LOT.eq(e.LOT) & detect.MINUTE.between(e.START_MIN, e.END_MIN - 1), "RULE_FLAG"] = True
detect["RULE_SCORED"] = [any(comparison[lot, s].N_PEERS.get(m, 0) >= 2 and pd.notna(comparison[lot, s].DELTA.get(m))
                             for s in NONCYCLIC) for lot, m in zip(detect.LOT, detect.MINUTE)]

# (2) 이동 Z-score: 같은 비주기 TC에서 이전 30분 대비 |Z| ≥ 4
z_columns = [s + "_Z30" for s in NONCYCLIC]
detect["Z_FLAG"] = detect[z_columns].abs().ge(Z_LIMIT).any(axis=1)
detect["Z_SCORED"] = detect[z_columns].notna().any(axis=1)

# (3) IsolationForest: 변화량 특징만 사용해 강종별 수준 차이를 이상으로 보지 않게 한다. 학습 8 LOT으로만 학습한다.
IF_COLUMNS = [s + suffix for s in IF_SIGNALS for suffix in ["_DEV30", "_DELTA1", "_PAST5_STD"]]
detect["IF_SCORED"] = detect[IF_COLUMNS].notna().all(axis=1)
fit_rows = detect.IF_SCORED & detect.LOT.isin(train_ids)
forest = IsolationForest(n_estimators=300, random_state=0).fit(detect.loc[fit_rows, IF_COLUMNS])
detect["IF_SCORE"] = np.nan
detect.loc[detect.IF_SCORED, "IF_SCORE"] = -forest.score_samples(detect.loc[detect.IF_SCORED, IF_COLUMNS])
IF_LIMIT = detect.loc[fit_rows, "IF_SCORE"].quantile(.99)          # 학습 구간 점수의 상위 1%
detect["IF_FLAG"] = detect.IF_SCORE.gt(IF_LIMIT)

# 분마다 알려진 상황: 사후 상태 → 실제 작업 → 강종변경 직후 30분 → 그 외 (앞의 것이 우선)
state_rows = pd.concat([pd.DataFrame({"LOT": k, "MINUTE": s.index, "STATE": s.to_numpy()}) for k, s in states.items()])
detect = detect.merge(state_rows, on=["LOT", "MINUTE"], how="left", validate="one_to_one")
detect["WORK"] = [any(m < b and m + 1 > a for a, b in work_ranges[lot]) for lot, m in zip(detect.LOT, detect.MINUTE)]
CONTEXTS = ["초기 승온", "종료 전환", "실제 작업 겹침", "강종변경 직후 30분", "그 외"]
detect["CONTEXT"] = np.select([detect.STATE.eq("초기 승온"), detect.STATE.eq("종료 전환"), detect.WORK, detect.MINUTE.lt(30)],
                              CONTEXTS[:4], default="그 외")
METHODS = {"RULE": "조건별 범위 기준선", "Z": "이동 Z-score", "IF": "IsolationForest"}

# 연속으로 표시된 분을 하나의 탐지 구간으로 묶는다. CP 후보는 기존 CP 규칙의 구간을 그대로 쓴다.
interval_rows = []
for key, name in METHODS.items():
    for lot, d in detect.groupby("LOT"):
        d = d.set_index("MINUTE")
        for a, b in runs(d[key + "_FLAG"]):
            block = d.loc[a:b - 1]
            if key == "RULE":
                sensors = episodes.loc[episodes.LOT.eq(lot) & episodes.START_MIN.lt(b) & episodes.END_MIN.gt(a), "SENSOR"].unique()
            else:
                strength = block[[s + "_Z30" for s in IF_SIGNALS]].abs().max().dropna().sort_values(ascending=False)
                sensors = [c.replace("_Z30", "") for c in strength.index[:2]]
            interval_rows.append({"METHOD": name, "LOT": lot, "START_MIN": a, "END_MIN": b, "MINUTES": b - a, "SENSORS": " · ".join(sensors)})
for c in cp_candidates.itertuples():
    interval_rows.append({"METHOD": "CP 규칙", "LOT": c.LOT, "START_MIN": int(np.floor(c.START_MIN)), "END_MIN": int(np.ceil(c.END_MIN)),
                          "MINUTES": int(np.ceil(c.END_MIN)) - int(np.floor(c.START_MIN)), "SENSORS": "CP-Z4"})
intervals = pd.DataFrame(interval_rows)
intervals["START"] = intervals.LOT.map(lot_meta.START) + pd.to_timedelta(intervals.START_MIN, unit="min")
intervals["END"] = intervals.LOT.map(lot_meta.START) + pd.to_timedelta(intervals.END_MIN, unit="min")
intervals["GRADE"] = intervals.LOT.map(lot_meta.GRADE)

In [ ]:
table(pd.DataFrame({"IF 기준 LOT":sorted(train_ids)}))

<a id="detector-locations"></a>

## 2. 세 탐지 방법은 무엇을 잡는가

정답 라벨이 없어 정확도를 계산할 수 없으므로, 각 방법이 **어떤 모양의 변화를 잡는지**를 실제 사례로 본다. 개수와 위치는 그림 아래 표에 있다.

In [ ]:
# 세 방법이 실제로 잡은 대표 사례를 한 장씩 보인다.
GC = {"SGCC": "#455a64", "SPFH590": "#d55e00", "SPCE": "#0072b2", "SPCEN": "#e69f00", "SPHC": "#009e73", "DP590": "#a23b57", "SPCD": "#7b2cbf"}
fig, axs = plt.subplots(1, 3, figsize=(30, 10))
fig.subplots_adjust(left=.04, right=.965, top=.7, bottom=.12, wspace=.2)
# (a) 범위 기준선: 같은 강종보다 Z7·Z8이 높은 LOT 하나
k, s = int(OVERAGE_LOTS[0]), "TC-Z7"; ax = axs[0]
for kk in [x for x in lots.LOT_NO.astype(int) if lot_meta.loc[x, "GRADE"] == lot_meta.loc[k, "GRADE"]]:
    y = profile[kk][(s, "median")]; ax.plot(y.index, y, color=SELECT_COLOR if kk == k else "#9aa6ae", lw=2.2 if kk == k else 1.1, label=f"{kk}" + (" (대상)" if kk == k else ""))
for r in intervals.loc[intervals.METHOD.eq("조건별 범위 기준선") & intervals.LOT.eq(k) & intervals.SENSORS.str.contains(s)].itertuples():
    ax.axvspan(r.START_MIN, r.END_MIN, color=SELECT_COLOR, alpha=.1, lw=0)
style(ax, "관측 시작 후 경과시간[분]"); ax.set_ylabel("온도[℃]"); ax.legend(fontsize=11.5, frameon=False, ncol=2, loc="lower right"); ax.set_xlim(0, lot_meta.loc[k, "MINUTES"] + 5)
gap = level_gap.set_index(["LOT", "SENSOR"]).loc[(k, s), "GAP"]
ax.set_title(f"(a) 범위 기준선 → '수준 차이'\n{k} {s}은 같은 {lot_meta.loc[k, 'GRADE']}보다 계속 {gap:+.1f}℃ (분홍 = 잡힌 구간)", loc="left", fontsize=15, pad=10)
# (b) 이동 Z: 예정된 변화·작업과 겹치지 않는 온도 구간 중 |Z|가 가장 큰 것
zc = intervals.loc[intervals.METHOD.eq("이동 Z-score") & intervals.SENSORS.str.startswith("TC")].copy()
zc = zc.loc[[not in_expected(int(r.LOT), r.START_MIN) and not any(r.START_MIN < y and r.END_MIN > x for x, y in work_ranges[int(r.LOT)]) for r in zc.itertuples()]]
zc["zmax"] = [detect.loc[detect.LOT.eq(r.LOT) & detect.MINUTE.between(r.START_MIN, r.END_MIN - 1), r.SENSORS.split(" · ")[0] + "_Z30"].abs().max() for r in zc.itertuples()]
r = zc.loc[zc.zmax.idxmax()]; s = r.SENSORS.split(" · ")[0]; k = int(r.LOT); ax = axs[1]
g = features.loc[features.LOT.eq(k)].set_index("MINUTE")[s + "_CURRENT"]
past = g.shift(1); center = past.rolling(Z_WINDOW, min_periods=Z_MIN).median(); spread = past.rolling(Z_WINDOW, min_periods=Z_MIN).std().clip(lower=SD_FLOOR["TC"])
a0, b0 = max(0, r.START_MIN - 45), r.END_MIN + 20
ax.fill_between(center.loc[a0:b0].index, (center - Z_LIMIT * spread).loc[a0:b0], (center + Z_LIMIT * spread).loc[a0:b0], color="#56b4e9", alpha=.18, label=f"직전 30분 기준 ±{Z_LIMIT} 표준편차")
ax.plot(center.loc[a0:b0].index, center.loc[a0:b0], color="#526d80", ls="--", lw=1.4, label="직전 30분 중앙값")
ax.plot(g.loc[a0:b0].index, g.loc[a0:b0], color="#17344d", lw=2, label=f"{k} {s}")
ax.axvspan(r.START_MIN, r.END_MIN, color="#0072b2", alpha=.2, lw=0)
style(ax, "관측 시작 후 경과시간[분]"); ax.set_ylabel("온도[℃]"); ax.legend(fontsize=11.5, frameon=False, loc="upper left")
ax.set_title(f"(b) 이동 Z-score → '갑작스러운 짧은 변화'\n{k} {s}: 직전 30분의 범위를 벗어난 {int(r.MINUTES)}분 (파랑 = 잡힌 구간)", loc="left", fontsize=15, pad=10)
# (c) IsolationForest: 계획수리 직전 정지
k = int(expected.loc[expected["종류"].eq("예정된 정지"), "LOT"].iloc[0]); ax = axs[2]; p = profile[k]; m = lot_meta.loc[k, "MINUTES"]
ax.plot(p.index, p.loc[:, [(f"TC-Z{z}", "median") for z in range(3, 7)]].mean(axis=1), color="#17344d", lw=2)
ax2 = ax.twinx(); ax2.plot(p.index, p.loc[:, [(f"OP-Z{z}", "median") for z in range(3, 7)]].median(axis=1), color="#d55e00", lw=1.5)
d = detect.loc[detect.LOT.eq(k) & detect.IF_FLAG]
for a, b in runs(pd.Series(True, index=d.MINUTE).reindex(range(int(m) + 1), fill_value=False)):
    ax.axvspan(a, b, color="#7b2cbf", alpha=.18, lw=0)
ax.set_xlim(m - 120, m + 1); style(ax, "관측 시작 후 경과시간[분]"); ax.set_ylabel("가열 TC-Z3~Z6 평균[℃]"); ax2.set_ylabel("가열 OP-Z3~Z6[%]", color="#d55e00")
ax2.spines[["top"]].set_visible(False)
ax.legend(handles=[Line2D([0], [0], color="#17344d", lw=2, label="가열 온도"), Line2D([0], [0], color="#d55e00", lw=1.5, label="가열 출력"),
                   Patch(facecolor="#7b2cbf", alpha=.3, label="잡힌 구간")], fontsize=11.5, frameon=False, loc="lower left")
ax.set_title(f"(c) IsolationForest → '끄고 켜는 큰 운전 변화'\n{k} 정기수리 전 정지: 온도·출력이 함께 떨어지는 끝부분 (보라 = 잡힌 구간)", loc="left", fontsize=15, pad=10)
header(fig, "세 탐지 방법은 서로 다른 것을 잡는다", "각 방법이 실제로 잡은 대표 사례 · 1분 중앙값 · 관측 시작 후 경과시간", [], ncol=1)
compact_header(fig, .92, .9, title_size=26, sub_size=15, legend_size=14)
axis_tag(fig, TAG_ELAPSED)
finish(fig, "01_탐지/01_세탐지_대표사례.png")
coverage=[]
for m in ['RULE','Z','IF']:
 coverage.append({'방법':METHODS[m],'계산 가능한 분':int(detect[m+'_SCORED'].sum()),'후보 분':int(detect[m+'_FLAG'].sum()),'후보 LOT':int(detect.loc[detect[m+'_FLAG'],'LOT'].nunique()),'구간':int(intervals.METHOD.eq(METHODS[m]).sum())})
table(pd.DataFrame(coverage));table(intervals)
agreement=[]
for m,n in [('RULE','Z'),('RULE','IF'),('Z','IF')]:agreement.append({'방법1':METHODS[m],'방법2':METHODS[n],'공통 후보 분':int((detect[m+'_FLAG']&detect[n+'_FLAG']).sum())})
table(pd.DataFrame(agreement))

<a id="evidence-scope"></a>

## 3. 근거로 좁히기 — 어느 LOT·구간을 봐야 하나

이 자료에는 설정값·통판 속도·두께·고장 기록이 없어 **원인(인과)을 확정할 수 없다.** 그래서 01~03에서 **이유를 이미 확인한 변화를 빼고**, 남은 것은 **나머지 데이터와 비교해 얼마나 떨어져 있는지**로 볼 곳을 정한다.

| 단계 | 하는 일 | 근거 |
|---|---|---|
| ① | 계획수리 직전 정지·직후 재가동 구간과 정지 앞 15분을 뺀다 | 바로 아래 그림: 모든 구역이 같은 방향으로 움직였고, 같은 크기의 변화는 이 4 LOT에만 있다 |
| ② | LOT 전체 수준 차이로 잡힌 범위 기준선 구간을 뺀다 | 03의 3절. 다만 과시효 Z7·Z8 수준은 아래에서 **같은 강종 LOT끼리 다시 비교**한다 |
| ③ | Z3 출력이 0%·100%에 붙었다 돌아오는 반복을 뺀다 | 03의 2절: 출력 100% 응답 31회 모두 60분 안에 원래 온도로 돌아왔고(중앙값 12분), 날짜가 지나도 길어지지 않았다 |
| CP | CP 규칙은 LOT마다 가장 큰 펄스 하나만 남기므로, **14 LOT 전체에서 CP가 벗어난 횟수와 크기를 모두 센다** | 아래 CP 그림 |

결과는 [4. 최종 선별](#final-focus)에 모은다.

### ① 예정된 정지·재가동을 빼는 근거

(a)는 14 LOT 전체에서 끝부분 하강·처음 상승이 큰 LOT을 찾고, (b)~(e)는 그 구간에서 모든 구역의 온도와 출력이 같은 방향으로 움직였는지 본다.

In [ ]:
# ① 예정된 정지·재가동을 빼는 근거
# (a) 14 LOT의 끝 5분 하강·처음 80~100분 상승 비교, (b)~(e) 정지·재가동 구간에서 구역별 온도·출력이 같은 방향으로 움직였는지
HEAT_OP = [f"OP-Z{z}" for z in range(3, 7)]
OVER = ["TC-Z7", "TC-Z8"]
role = dict(zip(expected.LOT, expected["수리 관계"]))
def med(p, cols):
    return p.loc[:, [(s, "median") for s in cols]]
judge_rows = []
for e in expected.itertuples():
    p = profile[e.LOT]
    if e.종류 == "예정된 정지":
        before, after = p.loc[e.START_MIN - 10:e.START_MIN - 1], p.loc[e.END_MIN - 5:]
    else:
        before, after = p.loc[:4], p.loc[e.END_MIN - 5:e.END_MIN - 1]
    heat = (med(after, HEAT).median() - med(before, HEAT).median()).to_numpy()
    over = (med(after, OVER).median() - med(before, OVER).median()).to_numpy()
    op0, op1 = med(before, HEAT_OP).median().median(), med(after, HEAT_OP).median().median()
    same = (heat < 0).all() and (over < 0).all() if e.종류 == "예정된 정지" else (heat > 0).all() and (over > 0).all()
    judge_rows.append({"LOT": e.LOT, "GRADE": e.GRADE, "종류": e.종류, "실제 시각": f"{e.시작:%m/%d %H:%M}–{e.끝:%H:%M}", "수리 관계": role[e.LOT],
                       "가열 Z3~Z6 온도 변화[℃]": f"{heat.min():+.1f} ~ {heat.max():+.1f}", "과시효 Z7·Z8 온도 변화[℃]": f"{over.min():+.1f} ~ {over.max():+.1f}",
                       "가열 출력 중앙값[%]": f"{op0:.0f} → {op1:.0f}", "판정": "모든 구역이 같은 방향 → 예정된 변화" if same else "다르게 움직인 구역 있음 → 확인 필요"})
stop_judge = pd.DataFrame(judge_rows)
level_rows = []
for k in lots.LOT_NO.astype(int):
    p = med(profile[k], HEAT); m = p.index.max()
    drop = (p.loc[m - 4:].median() - p.loc[m * .45:m * .6].median()).min()
    rise = (p.loc[80:100].median() - p.loc[:4].median()).max() if m >= 100 else np.nan
    level_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "끝 5분 하강[℃]": drop, "처음 80~100분 상승[℃]": rise})
end_rise = pd.DataFrame(level_rows).set_index("LOT")
big = end_rise.index[end_rise["끝 5분 하강[℃]"].le(-10) | end_rise["처음 80~100분 상승[℃]"].ge(20)].tolist()

fig = plt.figure(figsize=(30, 15))
gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1, 1], left=.085, right=.955, top=.83, bottom=.07, hspace=.62, wspace=.36)
ax = fig.add_subplot(gs[:, 0]); y = np.arange(len(end_rise))
ax.barh(y, end_rise["끝 5분 하강[℃]"], color="#0072b2", height=.6)
ax.barh(y, end_rise["처음 80~100분 상승[℃]"].fillna(0), color="#e69f00", height=.6)
for lim in [-10, 20]:
    ax.axvline(lim, color="#526d80", ls=":", lw=1.4)
ax.axvline(0, color="#9fb3c3", lw=1)
for yy, lo, hi in zip(y, end_rise["끝 5분 하강[℃]"], end_rise["처음 80~100분 상승[℃]"]):
    if lo <= -10: ax.text(lo - 1, yy, f"{lo:.0f}", ha="right", va="center", fontsize=13, color="#0072b2", fontweight="bold")
    if hi >= 20: ax.text(hi + 1, yy, f"+{hi:.0f}", ha="left", va="center", fontsize=13, color="#ac7500", fontweight="bold")
ax.set_yticks(y, [f"{k} · {g}" + (f"\n({role[k]})" if k in role else "") for k, g in zip(end_rise.index, end_rise.GRADE)])
ax.invert_yaxis(); ax.set_xlim(-58, 58); style(ax, "가열 TC-Z3~Z6 중 가장 크게 바뀐 구역[℃] · 점선 = -10℃ / +20℃"); ax.tick_params(labelsize=12)
ax.set_title("(a) 14 LOT: 끝 5분 하강(파랑) · 처음 80~100분 상승(노랑)\n" + f"점선을 넘는 LOT은 {', '.join(map(str, big))}뿐 — 모두 계획수리 바로 앞·뒤", loc="left", fontsize=15, pad=10)
for idx, e in enumerate(expected.itertuples()):
    a2 = fig.add_subplot(gs[idx // 2, 1 + idx % 2]); p = profile[e.LOT]; jr = stop_judge.iloc[idx]
    w0, w1 = (max(0, e.START_MIN - 40), lot_meta.loc[e.LOT, "MINUTES"]) if e.종류 == "예정된 정지" else (0, e.END_MIN + 30)
    seg = p.loc[w0:w1]; tx = lot_meta.loc[e.LOT, "START"] + pd.to_timedelta(seg.index, unit="min")
    h, o = med(seg, HEAT).mean(axis=1), med(seg, OVER).mean(axis=1)
    a2.plot(tx, h - h.iloc[0], color="#17344d", lw=2.2); a2.plot(tx, o - o.iloc[0], color="#8c6d31", lw=1.6, ls="--")
    b2 = a2.twinx(); b2.plot(tx, med(seg, HEAT_OP).median(axis=1), color="#d55e00", lw=1.3); b2.set_ylim(-5, 105)
    a2.axvspan(e.시작, e.끝, facecolor="none", edgecolor=EXPECTED_COLOR, hatch="///", lw=0)
    a2.set_title(f"({'bcde'[idx]}) {e.LOT} · {e.종류} ({e.시작:%m/%d})\n가열 {jr['가열 Z3~Z6 온도 변화[℃]']}℃ · 과시효 {jr['과시효 Z7·Z8 온도 변화[℃]']}℃ · 출력 {jr['가열 출력 중앙값[%]']}%", loc="left", fontsize=13.5, pad=8)
    clock_axis(a2); a2.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=2, maxticks=4)); a2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    a2.set_xlabel("실제 시각[시:분]", fontsize=11.5); a2.set_ylabel("왼쪽 끝 대비 온도 변화[℃]", fontsize=11.5)
    b2.set_ylabel("가열 출력[%]", fontsize=11.5, color="#d55e00"); a2.tick_params(labelsize=11); b2.tick_params(labelsize=11, colors="#d55e00"); b2.spines[["top"]].set_visible(False)
header(fig, "① 예정된 정지·재가동을 빼는 근거",
       "(a) 14 LOT 비교  |  (b)~(e) 정지·재가동 구간: 두 온도선은 그림 왼쪽 끝을 0으로 맞춘 변화, 주황 = 가열 OP-Z3~Z6 중앙값, 회색 빗금 = 예정된 변화 구간",
       [Line2D([0], [0], color="#17344d", lw=2.2, label="가열 TC-Z3~Z6 평균"), Line2D([0], [0], color="#8c6d31", lw=1.6, ls="--", label="과시효 TC-Z7·Z8 평균"),
        Line2D([0], [0], color="#d55e00", lw=1.3, label="가열 출력"), Patch(facecolor="white", edgecolor=EXPECTED_COLOR, hatch="///", label="예정된 정지·재가동")], ncol=4)
compact_header(fig, .935, .905, title_size=26, sub_size=14, legend_size=14)
axis_tag(fig, "(a) " + compare_tag("14 LOT") + "  |  (b)~(e) " + TAG_REAL)
finish(fig, "02_근거범위/01_예정된변화_근거.png")
table(stop_judge)
others = end_rise.drop(index=big)
restart_op = np.median([float(v.split(" → ")[1]) for v, kind in zip(stop_judge["가열 출력 중앙값[%]"], stop_judge["종류"]) if kind != "예정된 정지"])
note("\n".join([
    f"- **(a)** 끝 5분에 10℃ 넘게 내려가거나 처음 80~100분에 20℃ 넘게 오른 LOT은 {', '.join(map(str, big))}뿐이고, 모두 계획수리 바로 앞(마지막 관측)이나 뒤(첫 관측)다. "
    f"나머지 {len(others)} LOT은 끝 하강이 가장 커도 {others['끝 5분 하강[℃]'].min():+.1f}℃, 처음 상승이 가장 커도 {others['처음 80~100분 상승[℃]'].max():+.1f}℃다.",
    "- **(b)(d) 정지:** 가열·과시효 온도가 함께 내려가고, 가열 출력도 끝부분에서 0 쪽으로 내려간다(표의 가열 출력 중앙값).",
    f"- **(c)(e) 재가동:** 가열 출력은 평소 수준(약 {restart_op:.0f}%)을 유지한 채 가열·과시효 온도가 함께 올라간다. 온도만 혼자 오르거나 한 구역만 다르게 움직이지 않는다.",
    "- 표의 '판정'처럼 네 구간 모두 다르게 움직인 구역이 없다. 그래서 이 구간과 정지 앞 15분에 잡힌 탐지는 **예정된 변화**로 빼고, 이상 후보로 보지 않는다."]))


### 탐지 구간마다 이유 붙이기

126개 탐지 구간마다 ①~③ 가운데 어떤 이유로 설명되는지 붙인다. LOT마다 몇 개씩인지는 아래 둘째 표에 있다.

In [ ]:
# 탐지 구간마다 ①~③ 가운데 어떤 이유로 설명되는지 붙인다(앞 단계에서 설명되면 뒤 단계는 보지 않는다).
z_cols = [c for c in detect.columns if c.endswith("_Z30")]
level_pairs = set(zip(overage.LOT, overage.SENSOR))
def top_sensor(k, a, b):
    d = detect.loc[detect.LOT.eq(k) & detect.MINUTE.between(a, b - 1), z_cols].abs().max()
    return d.idxmax().replace("_Z30", "") if d.notna().any() else ""
def is_expected(r):
    return any(e.START_MIN - 15 < r.END_MIN and e.END_MIN > r.START_MIN for e in expected.loc[expected.LOT.eq(r.LOT)].itertuples())
def is_level(r):
    return r.METHOD == "조건별 범위 기준선" and all((r.LOT, s) in level_pairs or s == "CLN-IN" for s in r.SENSORS.split(" · "))
def is_z3(r):
    """이동 Z: 가장 크게 움직인 센서가 Z3 / 범위 기준선: TC-Z3·OP-Z3와 ②로 설명된 센서만 잡힘"""
    if r.METHOD == "CP 규칙":
        return False
    sens = set(r.SENSORS.split(" · ")); explained = {"TC-Z3", "OP-Z3", "CLN-IN"} | {s for kk, s in level_pairs if kk == r.LOT}
    return r.TOP_SENSOR in ["TC-Z3", "OP-Z3"] or (r.METHOD == "조건별 범위 기준선" and bool(sens & {"TC-Z3", "OP-Z3"}) and sens <= explained)
scope = intervals.copy()
scope["TOP_SENSOR"] = [top_sensor(k, a, b) for k, a, b in zip(scope.LOT, scope.START_MIN, scope.END_MIN)]
scope["이유"] = ["① 예정된 변화" if is_expected(r) else "② LOT 수준 차이" if is_level(r) else "③ Z3 반복" if is_z3(r) else "남음" for r in scope.itertuples()]
REASONS = ["① 예정된 변화", "② LOT 수준 차이", "③ Z3 반복", "남음"]
count = pd.crosstab(scope.METHOD, scope["이유"]).reindex(columns=REASONS, fill_value=0)
table(count.assign(합계=count.sum(axis=1)))
by_lot = pd.crosstab(scope.LOT, scope["이유"]).reindex(columns=REASONS, fill_value=0)
table(by_lot.assign(합계=by_lot.sum(axis=1)))
def nearby_work(k, a, b, hours=6):
    t0, t1 = lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(a)), lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(b))
    w = cal_work.loc[cal_work.STRT_DT.lt(t1 + pd.Timedelta(hours=hours)) & cal_work.END_DT.gt(t0 - pd.Timedelta(hours=hours))]
    return " / ".join(f"{x.MNT_TYPE} {x.STRT_DT:%m/%d %H:%M}–{x.END_DT:%H:%M} {remark_text(x)}" for x in w.itertuples()) or f"앞뒤 {hours}시간 작업 없음"
left = count["남음"]
note(f"- 126개 가운데 ① {count['① 예정된 변화'].sum()}개 · ② {count['② LOT 수준 차이'].sum()}개 · ③ {count['③ Z3 반복'].sum()}개는 이유가 확인된 변화다.\n"
     f"- 남은 {left.sum()}개는 CP 규칙 {left.get('CP 규칙', 0)}개와 이동 Z {left.get('이동 Z-score', 0)}개다. CP 규칙은 LOT마다 가장 큰 펄스 하나만 남기므로, CP는 아래에서 14 LOT 전체를 다시 센다.")


<a id="cp-all"></a>

### CP-Z4 — 14 LOT 원본에서 기준(±0.05)을 넘은 곳

CP-Z4가 **LOT 중앙값(모든 LOT 약 0.45)에서 0.05 넘게 벗어난 경우**를 1초 원본에서 모두 센다. 한 번은 중앙값에서 0.02 넘게 벗어나 있던 연속 구간이고, 크기는 그 구간에서 가장 크게 벗어난 값이다.

- **그림 읽는 법:** 한 줄이 한 LOT의 관측 전체다. 모든 줄이 같은 시간 눈금이라 관측 길이도 함께 비교된다. 회색 띠는 LOT 중앙값 ±0.05, 점은 벗어난 한 번의 꼭짓점이다(빨강 = 여러 번 벗어난 LOT, 주황 = 한 번). 오른쪽 끝은 횟수다.
- **세로 범위:** 작은 이탈도 보이게 ±0.2까지만 그렸다. 넘는 값은 숫자에 ↑·↓를 붙였다.
- **예정된 정지 중 변화**(240058 끝)는 빈 점으로만 두고 횟수에서 뺐다.
- 각 이탈의 실제 시각과 크기는 아래 표, 여러 번 벗어난 LOT의 파형은 이어지는 확대 그림에서 확인한다.

In [ ]:
# 14 LOT 전체에서 CP-Z4가 LOT 중앙값보다 0.05 넘게 벗어난 경우를 모두 센다(1초 원본).
WATCH = "#b8863b"   # 한 번 벗어난 LOT 색
cp_rows = []
for k in lots.LOT_NO.astype(int):
    s = raw[k].set_index("MEAS_DT")["CP-Z4"].dropna(); dev = s - s.median()
    off = dev.abs().gt(.02); grp = off.ne(off.shift()).cumsum()
    for _, part in dev[off].groupby(grp[off]):
        peak = part.abs().idxmax()
        if abs(part.loc[peak]) < .05:
            continue
        a, b, m = [(t - lot_meta.loc[k, "START"]).total_seconds() / 60 for t in (part.index[0], part.index[-1], peak)]
        cp_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "PEAK": peak, "DEV": part.loc[peak], "A": a, "B": b, "M": m,
                        "초": int((part.index[-1] - part.index[0]).total_seconds()) + 1, "EXP": in_expected(k, m)})
cp_events = pd.DataFrame(cp_rows)
cp_real = cp_events.loc[~cp_events.EXP]
cp_count = cp_real.groupby("LOT").size()
CP_FOCUS = cp_count[cp_count.ge(2)].sort_values(ascending=False).index.tolist(); CP_WATCH = cp_count[cp_count.eq(1)].index.tolist()
ids = lots.sort_values("START").LOT_NO.astype(int).tolist()
# 14 LOT의 CP-Z4 1초 원본을 같은 시간축에 한 줄씩 놓는다.
# 세로 = CP-Z4 - LOT 중앙값 · 회색 띠 = ±0.05 · 점 = 0.05 넘게 벗어난 한 번의 꼭짓점(빨강 = 여러 번 벗어난 LOT, 주황 = 한 번, 빈 점 = 예정된 정지 중)
# 세로는 ±0.2까지만 그린다(작은 이탈도 보이게). 넘는 값은 점을 끝에 두고 숫자에 ↑·↓를 붙인다.
fig, axs = plt.subplots(len(ids), 1, figsize=(28, 1.2 * len(ids) + 3.4), sharex=True, sharey=True)
H = 1.2 * len(ids) + 3.4; LIM = .2
fig.subplots_adjust(left=.13, right=.91, top=1 - 3.0 / H, bottom=.6 / H, hspace=.3)
for ax, k in zip(axs, ids):
    g = raw[k].set_index("MEAS_DT")["CP-Z4"].dropna(); dev = g - g.median()
    hrs = (g.index - lot_meta.loc[k, "START"]).total_seconds() / 3600
    ax.fill_between([0, hrs.max()], -.05, .05, color="#e3e9ee", lw=0)   # 띠는 관측한 시간까지만
    ax.plot(hrs, dev, color="#526d80", lw=.6)
    ev = cp_events.loc[cp_events.LOT.eq(k)]
    for r in ev.itertuples():
        c = EXPECTED_COLOR if r.EXP else SELECT_COLOR if k in CP_FOCUS else WATCH
        ax.plot(r.M / 60, np.clip(r.DEV, -LIM * .93, LIM * .93), "o", ms=9, mfc="white" if r.EXP else c, mec=c, mew=2, zorder=4)
    if len(ev):   # 가장 크게 벗어난 한 번만 값을 적는다
        r = ev.loc[ev.DEV.abs().idxmax()]
        arrow = "↑" if r.DEV > LIM else "↓" if r.DEV < -LIM else ""
        ax.annotate(("예정된 정지 중 " if r.EXP else "") + f"{r.DEV:+.3f}{arrow}", (r.M / 60, np.clip(r.DEV, -LIM * .93, LIM * .93)), xytext=(8, 0), textcoords="offset points", va="center", fontsize=12,
                    color=EXPECTED_COLOR if r.EXP else SELECT_COLOR if k in CP_FOCUS else WATCH, fontweight="bold" if k in CP_FOCUS else "normal")
    n = int(cp_count.get(k, 0)); col = SELECT_COLOR if k in CP_FOCUS else WATCH if n else "#9aa6ae"
    ax.text(-.01, .5, f"{k} · {lot_meta.loc[k, 'GRADE']} · {lot_meta.loc[k, 'START']:%m/%d}", transform=ax.transAxes, ha="right", va="center", fontsize=13,
            color=col if k in CP_FOCUS else "#17344d", fontweight="bold" if k in CP_FOCUS else "normal")
    ax.text(1.008, .5, f"{n}번" if n else "없음", transform=ax.transAxes, ha="left", va="center", fontsize=15, color=col, fontweight="bold" if k in CP_FOCUS else "normal")
    style(ax, ""); ax.set_yticks([-.05, .05], ["", ""]); ax.tick_params(labelsize=10.5); ax.set_ylim(-LIM, LIM)
    ax.spines["bottom"].set_visible(ax is axs[-1])
    if ax is not axs[-1]:
        ax.tick_params(axis="x", length=0)
axs[-1].set_xlim(0, lot_meta.MINUTES.max() / 60 + .2); axs[-1].set_xlabel("관측 시작 후 시간[시간] · 모든 줄이 같은 눈금이라 관측 길이도 함께 비교된다", fontsize=13)
axs[-1].tick_params(axis="x", labelsize=12.5)
fig.text(.91 + .008, 1 - 2.75 / H, "벗어난\n횟수", ha="left", va="bottom", fontsize=13, color="#526d80")
header(fig, "CP-Z4 · 14 LOT 원본에서 기준(±0.05)을 넘은 곳",
       "한 줄 = 한 LOT의 관측 전체(1초 원본) · 세로 = CP-Z4 - LOT 중앙값(모든 LOT 약 0.45) · 회색 띠 = ±0.05 · 점 = 0.05 넘게 벗어난 한 번의 꼭짓점 · 세로는 ±0.2까지(넘으면 숫자에 ↑↓) · 오른쪽 = 횟수(예정된 정지 중 제외)",
       [Line2D([0], [0], marker="o", ls="None", color=SELECT_COLOR, ms=11, label="여러 번 벗어난 LOT"), Line2D([0], [0], marker="o", ls="None", color=WATCH, ms=11, label="한 번 벗어난 LOT"),
        Line2D([0], [0], marker="o", ls="None", mfc="white", mec=EXPECTED_COLOR, mew=2, ms=11, label="예정된 정지 중 (횟수에서 뺌)"),
        Patch(facecolor="#e3e9ee", label="LOT 중앙값 ±0.05")], ncol=4)
compact_header(fig, 1 - 1.05 / H, 1 - 1.55 / H, title_size=25, sub_size=14, legend_size=13)
axis_tag(fig, "TIME AXIS · 관측 시작 후 시간[시간]")
finish(fig, "02_근거범위/03_CP이탈_14LOT.png")
table(cp_real.assign(시각=cp_real.PEAK.dt.strftime("%m/%d %H:%M:%S"), 크기=cp_real.DEV.round(3))[["LOT", "GRADE", "시각", "크기", "초"]].sort_values(["LOT", "시각"]))

In [ ]:
# 여러 번 벗어난 LOT: 위 = LOT 전체 CP-Z4, 아래 = 60분 안에 이어진 이탈을 한 묶음으로 1초 원본 확대(같은 구역 TC-Z4·OP-Z4 포함)
cp_card_rows = []
for k in CP_FOCUS:
    ev = cp_real.loc[cp_real.LOT.eq(k)].sort_values("PEAK"); groups = []
    for r in ev.itertuples():
        if groups and r.A - groups[-1][-1].B <= 60:
            groups[-1].append(r)
        else:
            groups.append([r])
    r0 = lot_meta.loc[k]; g = raw[k].set_index("MEAS_DT"); base = g["CP-Z4"].median()
    fig = plt.figure(figsize=(30, 16))
    gs = fig.add_gridspec(3, len(groups), height_ratios=[.8, 1, .62], left=.05, right=.955, top=.77, bottom=.06, hspace=.55, wspace=.25)
    top = fig.add_subplot(gs[0, :])
    top.plot(g.index, g["CP-Z4"], color=sensor_color("CP-Z4"), lw=.8)
    top.plot(ev.PEAK, ev.DEV + base, "o", color=SELECT_COLOR, ms=8, zorder=3)
    for x0, x1 in work_ranges[k]:
        top.axvspan(r0.START + pd.Timedelta(minutes=x0), r0.START + pd.Timedelta(minutes=x1), color=COL["WORK"], alpha=.2, lw=0)
    letters = "abcdef"
    for j, grp_ in enumerate(groups):
        t0, t1 = grp_[0].PEAK - pd.Timedelta(minutes=3), grp_[-1].PEAK + pd.Timedelta(minutes=6)
        top.axvspan(t0, t1, facecolor="none", edgecolor=SELECT_COLOR, lw=1.8)
        top.text(t1, .97, f" ({letters[j]})", transform=top.get_xaxis_transform(), ha="left", va="top", fontsize=13, color=SELECT_COLOR)
    top.set_xlim(r0.START, r0.END); clock_axis(top); top.set_ylabel("CP-Z4 기록값", fontsize=12); top.tick_params(labelsize=11.5)
    top.set_title(f"{k} 전체 CP-Z4 (1초) · 빨간 점 = 0.05 넘게 벗어난 꼭짓점 {len(ev)}개 · 빨간 상자 (a)(b)… = 아래 같은 글자의 확대", loc="left", fontsize=15, pad=10)
    for j, grp_ in enumerate(groups):
        w0 = max(0, grp_[0].A - 8); w1 = min(r0.MINUTES, grp_[-1].B + 8); w = raw_window(k, w0, w1)
        t_a = r0.START + pd.Timedelta(minutes=grp_[0].A); pre = w.loc[:t_a - pd.Timedelta(seconds=1)]
        c_ax = fig.add_subplot(gs[1, j]); t_ax = fig.add_subplot(gs[2, j], sharex=c_ax); o_ax = t_ax.twinx()
        c_ax.plot(w.index, w["CP-Z4"], color=sensor_color("CP-Z4"), lw=1.2)
        t_ax.plot(w.index, w["TC-Z4"] - pre["TC-Z4"].median(), color=sensor_color("TC-Z4"), lw=1.5)
        o_ax.plot(w.index, w["OP-Z4"], color="#526d80", lw=.9, alpha=.8); o_ax.set_ylim(-5, 105)
        for r in grp_:
            for a_ in [c_ax, t_ax]:
                a_.axvspan(r0.START + pd.Timedelta(minutes=r.A), r0.START + pd.Timedelta(minutes=r.B), color=SELECT_COLOR, alpha=.12, lw=0)
        during = w.loc[t_a:r0.START + pd.Timedelta(minutes=grp_[-1].B)]
        spread = lambda x, c: (x - c).abs().max()
        dtc, ptc = spread(during["TC-Z4"], pre["TC-Z4"].median()), spread(pre["TC-Z4"], pre["TC-Z4"].median())
        dop, pop = spread(during["OP-Z4"], pre["OP-Z4"].median()), spread(pre["OP-Z4"], pre["OP-Z4"].median())
        top_dev = max((r.DEV for r in grp_), key=abs)
        cp_card_rows.append({"LOT": k, "묶음": f"({letters[j]})", "시각": f"{grp_[0].PEAK:%m/%d %H:%M}–{grp_[-1].PEAK:%H:%M}", "이탈 횟수": len(grp_), "최대 크기": round(top_dev, 3),
                             "TC-Z4 변화[℃]": round(dtc, 1), "TC-Z4 앞 10분 움직임[℃]": round(ptc, 1), "OP-Z4 변화[%p]": round(dop, 0), "OP-Z4 앞 10분 움직임[%p]": round(pop, 0)})
        c_ax.set_title(f"({letters[j]}) {grp_[0].PEAK:%m/%d %H:%M}–{grp_[-1].PEAK:%H:%M} · {len(grp_)}번 · 최대 {top_dev:+.3f}", loc="left", fontsize=14.5, pad=8)
        t_ax.set_title(f"같은 시각 TC-Z4 최대 {dtc:.1f}℃ (앞 10분 {ptc:.1f}℃) · OP-Z4 최대 {dop:.0f}%p (앞 10분 {pop:.0f}%p)", loc="left", fontsize=12.5, pad=6, color="#526d80")
        style(c_ax, ""); c_ax.set_ylabel("CP-Z4 기록값", fontsize=11.5); c_ax.tick_params(labelbottom=False, labelsize=11)
        clock_axis(t_ax); t_ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=7)); t_ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        t_ax.set_xlabel("실제 시각[시:분]", fontsize=11); t_ax.set_ylabel("TC-Z4 변화[℃]", fontsize=11); t_ax.tick_params(labelsize=10.5)
        o_ax.set_ylabel("OP-Z4[%]", fontsize=11, color="#526d80"); o_ax.tick_params(labelsize=10.5, colors="#526d80"); o_ax.spines[["top"]].set_visible(False)
        t_ax.set_xlim(w.index[0], w.index[-1])
    header(fig, f"{k} · {r0.GRADE} · CP-Z4가 {len(ev)}번 벗어남", "위 = LOT 전체 · 아래 = 묶음별 1초 원본: CP-Z4와 같은 구역 TC-Z4(앞 10분 중앙값 대비)·OP-Z4 · 빨간 음영 = 벗어난 구간 · 갈색 = 실제 작업",
           [Line2D([0], [0], color=sensor_color("CP-Z4"), lw=1.4, label="CP-Z4"), Line2D([0], [0], color=sensor_color("TC-Z4"), lw=1.6, label="TC-Z4 변화"),
            Line2D([0], [0], color="#526d80", lw=1, label="OP-Z4"), Patch(facecolor=SELECT_COLOR, alpha=.25, label="벗어난 구간")], ncol=4)
    compact_header(fig, .925, .89, title_size=25, sub_size=14, legend_size=13)
    axis_tag(fig, TAG_REAL)
    finish(fig, f"02_근거범위/04_CP반복_{k}.png")
cp_card = pd.DataFrame(cp_card_rows); table(cp_card)
after_work = {k: cal_work.loc[cal_work.DETC_DT.ge(lot_meta.loc[k, "END"]) & cal_work.DETC_DT.lt(lot_meta.loc[k, "END"] + pd.Timedelta(hours=12))] for k in CP_FOCUS}
note("\n".join(
    [f"- **{k}:** CP-Z4가 {int(cp_count[k])}번 벗어났고 최대 {cp_real.loc[cp_real.LOT.eq(k), 'DEV'].abs().max():.3f}이다. "
     + " · ".join(f"{r['묶음']} {r['시각']} {r['이탈 횟수']}번(TC-Z4 {r['TC-Z4 변화[℃]']}℃, 앞 10분 {r['TC-Z4 앞 10분 움직임[℃]']}℃)" for r in cp_card_rows if r["LOT"] == k)
     + (". 관측 끝 뒤 12시간 안 정비 감지: " + " / ".join(f"{w.MNT_TYPE} {w.DETC_DT:%m/%d %H:%M} 감지({remark_text(w)})" for w in after_work[k].itertuples()) if len(after_work[k]) else ". 관측 끝 뒤 12시간 안 정비 감지 없음")
     for k in CP_FOCUS] +
    ["- CP가 평소 값(0.45)의 수십 %에서 두 배 가까이 뛰는 동안, 같은 구역 TC-Z4는 1℃ 안팎만 움직였다. 노 안 온도 변화로는 이 크기를 설명하기 어렵다. **CP 측정·기록(센서·전송·계산식)을 먼저 확인한다.** 시간 순서만으로 정비와의 인과는 정하지 않는다."]))


<a id="level-compare"></a>

### 과시효 Z7·Z8 — 같은 강종 LOT끼리 나란히 보기

②에서 뺀 LOT 수준 차이 가운데 과시효 Z7·Z8은 LOT 전체가 다른 경우라 따로 본다. 같은 강종의 LOT을 나란히 놓고, **안정 참고 구간 1분 중앙값의 분포(상자그림)**를 비교한다. 상자 위 숫자는 LOT 중앙값이다.

- **혼자 떨어짐:** LOT이 3개 이상인 강종에서, 이 LOT과 가장 가까운 LOT의 중앙값 차이가 **다른 LOT끼리의 중앙값 차이보다 큰** 경우
- **LOT 2개뿐:** 비교 대상이 하나라 어느 쪽이 다른지 정할 수 없다

In [ ]:
# 같은 강종 LOT의 과시효 Z7·Z8 안정 참고 수준을 나란히 놓는다.
def stable_values(k, s):
    return profile[k][(s, "median")].where(quality[k][s].ge(.9) & states[k].eq("안정 참고")).dropna()
grade_lots = {g: d.sort_values("START").LOT_NO.astype(int).tolist() for g, d in lots.groupby("GRADE") if len(d) >= 2}
grade_order = sorted(grade_lots, key=lambda g: -len(grade_lots[g]))
rows = []
for g in grade_order:
    ks = grade_lots[g]
    for s in OVER:
        meds = {k: stable_values(k, s).median() for k in ks}
        for k in ks:
            others = [meds[x] for x in ks if x != k]
            near = min(abs(meds[k] - o) for o in others); spread = max(others) - min(others) if len(others) >= 2 else np.nan
            rows.append({"GRADE": g, "LOT": k, "SENSOR": s, "중앙값[℃]": meds[k], "가장 가까운 LOT과 차이[℃]": near, "다른 LOT끼리 차이[℃]": spread,
                         "판정": "LOT 2개뿐" if len(ks) == 2 else "혼자 떨어짐" if near > spread else ""})
lot_level = pd.DataFrame(rows)
apart = lot_level.loc[lot_level["판정"].eq("혼자 떨어짐")].groupby("LOT").SENSOR.nunique()
LEVEL_FOCUS = apart[apart.eq(len(OVER))].index.tolist()
usual_gap = lot_level.loc[~lot_level.LOT.isin(LEVEL_FOCUS) & lot_level["판정"].ne("LOT 2개뿐")].groupby(["GRADE", "SENSOR"])["중앙값[℃]"].agg(lambda v: v.max() - v.min()).max()
pair_gap = lot_level.loc[lot_level["판정"].eq("LOT 2개뿐")].groupby(["GRADE", "SENSOR"])["중앙값[℃]"].agg(lambda v: v.max() - v.min()).unstack()
LEVEL_PAIR = [g for g in pair_gap.index if pair_gap.loc[g].min() > usual_gap]
LEVEL_WATCH = [k for g in LEVEL_PAIR for k in grade_lots[g]]
fig = plt.figure(figsize=(30, 14))
gs = fig.add_gridspec(2, len(grade_order), width_ratios=[len(grade_lots[g]) + .6 for g in grade_order], left=.05, right=.98, top=.78, bottom=.08, hspace=.5, wspace=.22)
for row, s in enumerate(OVER):
    for col, g in enumerate(grade_order):
        ax = fig.add_subplot(gs[row, col]); ks = grade_lots[g]
        vals = [stable_values(k, s) for k in ks]
        bp = ax.boxplot(vals, widths=.55, showfliers=False, patch_artist=True, medianprops=dict(color="#17344d", lw=2))
        for patch, k in zip(bp["boxes"], ks):
            patch.set_facecolor(SELECT_COLOR if k in LEVEL_FOCUS else WATCH if k in LEVEL_WATCH else "#d5dce1"); patch.set_alpha(.75); patch.set_edgecolor("#526d80")
        lo = min(v.quantile(.25) - 1.5 * (v.quantile(.75) - v.quantile(.25)) for v in vals); hi = max(v.quantile(.75) + 1.5 * (v.quantile(.75) - v.quantile(.25)) for v in vals)
        pad = (hi - lo) * .12 + .3
        for x_, v, k in zip(range(1, len(ks) + 1), vals, ks):
            ax.text(x_, min(v.max(), v.quantile(.75) + 1.5 * (v.quantile(.75) - v.quantile(.25))) + pad * .25, f"{v.median():.1f}", ha="center", va="bottom", fontsize=12.5,
                    color=SELECT_COLOR if k in LEVEL_FOCUS else "#ac7500" if k in LEVEL_WATCH else "#17344d", fontweight="bold" if k in LEVEL_FOCUS + LEVEL_WATCH else "normal")
        ax.set_ylim(lo - pad * .3, hi + pad)
        ax.set_xticks(range(1, len(ks) + 1), [f"{k}\n{lot_meta.loc[k, 'START']:%m/%d}" for k in ks])
        style(ax, ""); ax.set_ylabel(f"{s}[℃]" if col == 0 else "", fontsize=12.5); ax.tick_params(labelsize=11.5)
        d = lot_level.loc[lot_level.GRADE.eq(g) & lot_level.SENSOR.eq(s)].set_index("LOT")
        f = [k for k in ks if k in LEVEL_FOCUS]
        verdict = (f"{f[0]}만 {d.loc[f[0], '가장 가까운 LOT과 차이[℃]']:.1f}℃ 떨어짐" if f else
                   f"2 LOT뿐 · 차이 {pair_gap.loc[g, s]:.1f}℃" + (" (판단 불가)" if g in LEVEL_PAIR else "") if len(ks) == 2 else "혼자 떨어진 LOT 없음")
        ax.set_title(f"({'abcdefgh'[row * len(grade_order) + col]}) {g} · {s}\n{verdict}", loc="left", fontsize=13.5, pad=8)
header(fig, "과시효 Z7·Z8 · 같은 강종 LOT끼리 비교", "상자 = 안정 참고 구간 1분 중앙값의 분포(수염 1.5×IQR) · 숫자 = LOT 중앙값 · 빨강 = 혼자 떨어진 LOT · 노랑 = LOT 2개뿐이라 어느 쪽인지 판단 불가",
       [Patch(facecolor=SELECT_COLOR, alpha=.75, label="혼자 떨어진 LOT"), Patch(facecolor=WATCH, alpha=.75, label="2 LOT뿐 · 판단 불가"), Patch(facecolor="#d5dce1", label="같은 강종 안")], ncol=3)
compact_header(fig, .925, .89, title_size=25, sub_size=14, legend_size=13)
axis_tag(fig, compare_tag("같은 강종 LOT"))
finish(fig, "02_근거범위/05_과시효Z7Z8_같은강종비교.png")
table(lot_level.round(2))
op_gap = level_gap.loc[level_gap.LOT.isin(LEVEL_FOCUS + LEVEL_WATCH) & level_gap.SENSOR.isin(["OP-Z7", "OP-Z8"])].pivot(index="LOT", columns="SENSOR", values="GAP")
note("\n".join(
    [f"- **{k} ({lot_meta.loc[k, 'GRADE']}):** Z7·Z8 중앙값이 같은 강종에서 가장 가까운 LOT보다 "
     f"{lot_level.loc[lot_level.LOT.eq(k) & lot_level.SENSOR.eq('TC-Z7'), '가장 가까운 LOT과 차이[℃]'].iloc[0]:.1f}℃·{lot_level.loc[lot_level.LOT.eq(k) & lot_level.SENSOR.eq('TC-Z8'), '가장 가까운 LOT과 차이[℃]'].iloc[0]:.1f}℃ 떨어져 있고, "
     f"다른 LOT끼리는 {lot_level.loc[lot_level.LOT.eq(k), '다른 LOT끼리 차이[℃]'].max():.1f}℃ 안이다. 출력은 같은 강종 대비 OP-Z7 {op_gap.loc[k, 'OP-Z7']:+.1f}%p · OP-Z8 {op_gap.loc[k, 'OP-Z8']:+.1f}%p다."
     for k in LEVEL_FOCUS] +
    [f"- **{' · '.join(map(str, grade_lots[g]))} ({g}):** 두 LOT의 차이는 Z7 {pair_gap.loc[g, 'TC-Z7']:.1f}℃ · Z8 {pair_gap.loc[g, 'TC-Z8']:.1f}℃로, 혼자 떨어진 LOT을 뺀 같은 강종 LOT끼리의 차이(최대 {usual_gap:.1f}℃)보다 크다. "
     "하지만 LOT이 2개뿐이라 **어느 쪽이 다른지 정할 수 없다.** 두 LOT 모두 지켜볼 것에 둔다." for g in LEVEL_PAIR] +
    ["- 온도는 높은데 출력은 같거나 낮다. 설정 온도가 달랐는지, 열전대가 치우쳤는지는 **설정값·교정 기록**으로 확인해야 한다."]))


### 240108 짧은 변화 — 확대해 보니 설명됨

①~③ 뒤에 남은 이동 Z 구간 1개다. 1초 원본과 LOT 전체 흐름을 함께 본다.

In [ ]:
# ①~③ 뒤에 남은 이동 Z 구간: LOT 전체에서의 위치와 1초 원본
remain_z = scope.loc[scope["이유"].eq("남음") & scope.METHOD.ne("CP 규칙")]
for r in remain_z.itertuples():
    k, a, b = int(r.LOT), float(r.START_MIN), float(r.END_MIN); r0 = lot_meta.loc[k]; sens = r.SENSORS.split(" · ")
    panels = sens + [s.replace("OP-", "TC-") for s in sens if s.startswith("OP-") and s.replace("OP-", "TC-") not in sens]
    d = detect.loc[detect.LOT.eq(k) & detect.MINUTE.between(a, b - 1)]; zmax = d[[s + "_Z30" for s in sens]].abs().max().max()
    w = raw_window(k, max(0, a - 15), min(r0.MINUTES, b + 15)); t_a, t_b = r0.START + pd.Timedelta(minutes=a), r0.START + pd.Timedelta(minutes=b)
    fig = plt.figure(figsize=(30, 12))
    gs = fig.add_gridspec(2, len(panels), height_ratios=[.75, 1.5], left=.05, right=.97, top=.75, bottom=.08, hspace=.55, wspace=.22)
    top = fig.add_subplot(gs[0, :]); s0 = sens[0]; p = profile[k][(s0, "median")]
    top.plot(r0.START + pd.to_timedelta(p.index, unit="min"), p, color=sensor_color(s0), lw=1.3)
    for x0, x1 in work_ranges[k]:
        top.axvspan(r0.START + pd.Timedelta(minutes=x0), r0.START + pd.Timedelta(minutes=x1), color=COL["WORK"], alpha=.18, lw=0)
    top.axvspan(t_a - pd.Timedelta(minutes=4), t_b + pd.Timedelta(minutes=4), fill=False, ec=SELECT_COLOR, lw=2.5)
    top.set_xlim(r0.START, r0.END); clock_axis(top); top.set_ylabel(shown_unit(s0), fontsize=12); top.tick_params(labelsize=11.5)
    top.set_title(f"(a) {k} 전체 · {s0} 1분 중앙값 · 빨간 상자 = 아래 확대 위치 · 갈색 음영 = 실제 작업", loc="left", fontsize=14.5, pad=8)
    lines = []
    for j, s in enumerate(panels):
        ax = fig.add_subplot(gs[1, j]); v = w[s] * sensor_scale(s)
        ax.plot(w.index, v, color=sensor_color(s), lw=1.4); ax.axvspan(t_a, t_b, color=SELECT_COLOR, alpha=.12, lw=0)
        pre, dur = v.loc[:t_a - pd.Timedelta(seconds=1)].tail(600).median(), v.loc[t_a:t_b]
        lines.append(f"{s} 앞 10분 {pre:.1f} → 구간 중 {dur.min():.1f}~{dur.max():.1f}")
        ax.set_title(f"({'bcdef'[j]}) {s}: 앞 10분 중앙값 {pre:.1f} → 구간 중 {dur.min():.1f}~{dur.max():.1f}", loc="left", fontsize=13.5, pad=8)
        ax.set_xlim(w.index[0], w.index[-1]); clock_axis(ax); ax.xaxis.set_major_locator(mdates.MinuteLocator(byminute=range(0, 60, 5))); ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        ax.set_xlabel("실제 시각[시:분]", fontsize=11.5); ax.set_ylabel(shown_unit(s), fontsize=11.5); ax.tick_params(labelsize=11)
    header(fig, f"남은 짧은 변화 · {k} · {r0.GRADE} · 최대 |Z| {zmax:.1f} · {int(r.MINUTES)}분", f"{when(k, a, b)} · {r.METHOD}가 잡았고 ①~③ 어느 이유와도 맞지 않음 · 빨간 음영 = 잡힌 구간 · 1초 원본",
           [Patch(facecolor=SELECT_COLOR, alpha=.25, label="잡힌 구간"), Patch(facecolor=COL["WORK"], alpha=.3, label="실제 작업")], ncol=2)
    compact_header(fig, .925, .89, title_size=25, sub_size=14, legend_size=13)
    axis_tag(fig, TAG_REAL)
    finish(fig, f"02_근거범위/06_남은_짧은변화_{k}.png")
    d1m = p.diff(); here = d1m.loc[a:b - 1]; here = here.loc[here.abs().idxmax()]
    note(f"**{k} · {when(k, a, b)}:** (a)에서 보듯 {s0} 값은 이 LOT 동안 {p.min():.1f}~{p.max():.1f} 사이를 천천히 오르내린다. 잡힌 1분은 그 가운데 한 하강에서 가장 가파른 1분"
         f"({here:+.2f}/분, LOT 안 1분 변화 크기 {int((d1m.abs() > abs(here)).sum()) + 1}위)이고, 최대 |Z|({zmax:.1f})는 기준({Z_LIMIT})과 거의 같다. "
         + " / ".join(lines) + f". 앞뒤 6시간 작업: {nearby_work(k, a, b)}. **반복 파형으로 설명되는 변화라 이상 후보에서 뺀다.**")
EXPLAINED_SHORT = remain_z.LOT.astype(int).tolist()


<a id="final-focus"></a>

## 4. 최종: 확실히 볼 LOT·구간

근거로 좁힌 결과 **확실히 볼 LOT은 4개**다. LOT마다 한 줄씩 놓았다.

- **첫 그림 (우리가 볼 그림):** 왼쪽 = LOT·강종·관측 시각·공정과 센서, 가운데 = 근거 그림, 오른쪽 = 무엇이 보였나·근거·다음에 확인할 기록.
- **둘째 그림 (보고서용):** 같은 그림에서 오른쪽 설명 칸만 뺐다.
- **CP-Z4 (240069·240037):** 14 LOT 가운데 0.05 넘게 **여러 번** 벗어난 LOT은 이 둘뿐이다. 같은 시각 같은 구역 온도 TC-Z4는 1℃ 안팎만 움직였다.
- **과시효 Z7·Z8 (240091·240121):** 같은 강종 다른 LOT끼리는 0.6℃ 안에 모여 있는데 이 LOT만 약 3℃ 떨어져 있다. 출력은 같거나 낮다.
- 그림 아래 표는 **지켜볼 것**과 **설명됨**이다.

"확실히 볼 것"은 **확인 우선순위**다. 고장이나 정비 효과가 확정됐다는 뜻은 아니다.

In [ ]:
# 최종 선별: 확실히 볼 LOT마다 한 줄. 같은 근거 그림으로 두 장을 저장한다.
#   01 = 우리가 볼 그림(오른쪽에 무엇이 보였나·근거·다음 확인 기록) · 02 = 보고서용(오른쪽 설명 칸 없음)
focus = [("CP", k) for k in CP_FOCUS] + [("LEVEL", k) for k in LEVEL_FOCUS]
n, H = len(focus), 4.3 * len(focus) + 3.2
INK, SUB = "#17344d", "#526d80"
span = lambda t: t.split("–")[0] if t.split("–")[0][-5:] == t.split("–")[1] else t   # 한 번뿐이면 시작 시각만
pp = lambda v: f"{0 if abs(v) < .05 else v:+.1f}"   # -0.0 대신 +0.0

def evidence(mid, kind, k):
    """가운데 근거 그림을 그리고, 오른쪽에 쓸 설명 글을 돌려준다."""
    r0 = lot_meta.loc[k]
    if kind == "CP":
        g = raw[k].set_index("MEAS_DT")["CP-Z4"]; base = g.median(); ev = cp_real.loc[cp_real.LOT.eq(k)]
        mid.axhspan(base - .05, base + .05, color="#e8edf1", lw=0)
        mid.plot(g.index, g, color=sensor_color("CP-Z4"), lw=.8)
        mid.plot(ev.PEAK, ev.DEV + base, "o", color=SELECT_COLOR, ms=8, zorder=3)
        mid.set_xlim(r0.START, r0.END); mid.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=5)); mid.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
        style(mid, ""); mid.set_ylabel("CP-Z4", fontsize=12); mid.tick_params(labelsize=11.5)
        mid.set_title(f"LOT 전체 CP-Z4 (1초) · 회색 띠 = 평소 ±0.05 · 빨간 점 = 벗어난 꼭짓점 {len(ev)}개", loc="left", fontsize=13.5, pad=8)
        groups = [c for c in cp_card_rows if c["LOT"] == k]
        tc_max = max(c["TC-Z4 변화[℃]"] for c in groups); tc_pre = max(c["TC-Z4 앞 10분 움직임[℃]"] for c in groups)
        when_txt = " · ".join(f"{span(c['시각'])} {c['이탈 횟수']}번" for c in groups)
        aw = after_work[k]
        nxt = "CP-Z4 센서·전송·계산 기록 (" + ", ".join(span(c["시각"]) for c in groups) + ")"
        if len(aw):
            nxt += "\n관측 끝 뒤 정비 감지: " + " / ".join(f"{w.MNT_TYPE} {w.DETC_DT:%m/%d %H:%M}({remark_text(w)})" for w in aw.itertuples()) + " — 시간 순서만 기록"
        return [("무엇이 보였나", f"CP-Z4가 평소(약 {base:.2f})에서 0.05 넘게 {len(ev)}번 벗어남\n{when_txt} · 최대 {ev.DEV.abs().max():+.3f}"),
                ("근거", f"14 LOT 중 여러 번 벗어난 LOT은 {len(CP_FOCUS)}개뿐 (나머지는 0~1번)\n같은 시각 TC-Z4는 최대 {tc_max:.1f}℃ (앞 10분 평소 움직임 {tc_pre:.1f}℃)"),
                ("다음 확인 기록", nxt)]
    g = r0.GRADE; ks = grade_lots[g]
    for yy, s in enumerate(OVER):
        d = lot_level.loc[lot_level.SENSOR.eq(s) & lot_level.GRADE.eq(g)].set_index("LOT")
        others = d.loc[[kk for kk in ks if kk != k], "중앙값[℃]"]
        mid.plot(others, [yy] * len(others), "o", ms=12, color="#9aa6ae", zorder=3)
        mid.annotate(f"다른 {len(others)} LOT {others.min():.1f}~{others.max():.1f}", (others.mean(), yy), xytext=(0, -14), textcoords="offset points",
                     ha="center", va="top", fontsize=11.5, color=SUB)
        mid.plot(d.loc[k, "중앙값[℃]"], yy, "o", ms=16, color=SELECT_COLOR, zorder=4)
        mid.annotate(f"{k} {d.loc[k, '중앙값[℃]']:.1f}", (d.loc[k, "중앙값[℃]"], yy), xytext=(0, 14), textcoords="offset points",
                     ha="center", va="bottom", fontsize=12, color=SELECT_COLOR, fontweight="bold")
    allv = lot_level.loc[lot_level.GRADE.eq(g), "중앙값[℃]"]
    mid.set_xlim(allv.min() - 1, allv.max() + 1); mid.set_ylim(-.9, 1.9)
    mid.set_yticks([0, 1], ["TC-Z7", "TC-Z8"]); style(mid, "안정 참고 구간 중앙값[℃]"); mid.tick_params(labelsize=12)
    mid.set_title(f"같은 강종 {g} {len(ks)} LOT의 과시효 온도 (빨강 = 이 LOT, 회색 = 같은 강종 다른 LOT)", loc="left", fontsize=13.5, pad=8)
    d7 = lot_level.loc[lot_level.LOT.eq(k) & lot_level.SENSOR.eq("TC-Z7")].iloc[0]; d8 = lot_level.loc[lot_level.LOT.eq(k) & lot_level.SENSOR.eq("TC-Z8")].iloc[0]
    return [("무엇이 보였나", f"LOT 전체에서 과시효 Z7·Z8이 같은 강종 다른 LOT보다 높음\nZ7 +{d7['가장 가까운 LOT과 차이[℃]']:.1f}℃ · Z8 +{d8['가장 가까운 LOT과 차이[℃]']:.1f}℃ (가장 가까운 LOT 기준)"),
            ("근거", f"같은 강종 다른 LOT끼리는 {max(d7['다른 LOT끼리 차이[℃]'], d8['다른 LOT끼리 차이[℃]']):.1f}℃ 안에 모여 있고 이 LOT만 떨어져 있음\n출력은 OP-Z7 {pp(op_gap.loc[k, 'OP-Z7'])}%p · OP-Z8 {pp(op_gap.loc[k, 'OP-Z8'])}%p로 오히려 같거나 낮음"),
            ("다음 확인 기록", "Z7·Z8 설정 온도와 열전대 교정 기록\n같은 강종 다른 LOT 날짜(" + ", ".join(f"{kk} {lot_meta.loc[kk, 'START']:%m/%d}" for kk in ks if kk != k) + ")와 비교")]

def final_cards(report, name):
    ratios, width, L, R = ([.62, 1.9], 23, .025, .985) if report else ([.62, 1.55, 1.5], 30, .02, .99)
    fig = plt.figure(figsize=(width, H))
    gs = fig.add_gridspec(n, len(ratios), width_ratios=ratios, left=L, right=R, top=1 - (2.1 if report else 2.7) / H, bottom=.05, hspace=.55, wspace=.06)
    for row, (kind, k) in enumerate(focus):
        r0 = lot_meta.loc[k]
        lt = fig.add_subplot(gs[row, 0]); lt.axis("off")
        lt.text(0, .92, f"{k}", fontsize=26, fontweight="bold", color=SELECT_COLOR, va="top", transform=lt.transAxes)
        lt.text(0, .62, f"{r0.GRADE} · {r0.START:%m/%d %H:%M}~{r0.END:%m/%d %H:%M}", fontsize=14, color=INK, va="top", transform=lt.transAxes)
        proc = "가열·균열 Z4\nCP-Z4 (무차원 기록값)" if kind == "CP" else "과시효 Z7·Z8\nTC-Z7 · TC-Z8 [℃]"
        lt.text(0, .42, proc, fontsize=14, color=SUB, va="top", transform=lt.transAxes, linespacing=1.5)
        lines = evidence(fig.add_subplot(gs[row, 1]), kind, k)
        if not report:
            rt = fig.add_subplot(gs[row, 2]); rt.axis("off"); y = 1.0
            for head, body in lines:
                rt.text(0, y, head, fontsize=13, color=SUB, fontweight="bold", va="top", transform=rt.transAxes)
                rt.text(0, y - .1, body, fontsize=13.5, color=INK, va="top", transform=rt.transAxes, linespacing=1.4)
                y -= .1 + .095 * len(body.splitlines()) + .08
        if row < n - 1:
            fig.add_artist(Line2D([L, R], [gs[row, 0].get_position(fig).y0 - .018] * 2, color="#d5dce1", lw=1, transform=fig.transFigure))
    sub = "" if report else "LOT마다 한 줄 · 왼쪽 = LOT·강종·관측 시각·공정과 센서 · 가운데 = 근거 그림 · 오른쪽 = 무엇이 보였나 · 근거 · 다음에 확인할 기록"
    header(fig, f"최종 선별 · 확실히 볼 LOT {n}개", sub, [], ncol=1)
    compact_header(fig, 1 - 1.15 / H, 1 - 1.6 / H, title_size=27, sub_size=14.5)
    axis_tag(fig, "CP: 실제 시각 | 과시효: 같은 강종 비교")
    finish(fig, name)

final_cards(False, "04_최종/01_최종선별_확실히볼LOT.png")
final_cards(True, "04_최종/02_최종선별_보고서용.png")
watch_rows = [{"구분": "CP 한 번", "LOT": k, "강종": lot_meta.loc[k, "GRADE"], "공정·센서": "가열·균열 Z4 · CP-Z4",
               "무엇": f"{cp_real.loc[cp_real.LOT.eq(k), 'PEAK'].iloc[0]:%m/%d %H:%M} · {cp_real.loc[cp_real.LOT.eq(k), 'DEV'].iloc[0]:+.3f}",
               "왜 지켜보나": "한 번뿐이라 반복되는지 아직 알 수 없음"} for k in CP_WATCH]
watch_rows += [{"구분": "Z7·Z8 2 LOT", "LOT": " · ".join(map(str, grade_lots[g])), "강종": g, "공정·센서": "과시효 TC-Z7·Z8",
                "무엇": f"두 LOT 차이 Z7 {pair_gap.loc[g, 'TC-Z7']:.1f}℃ · Z8 {pair_gap.loc[g, 'TC-Z8']:.1f}℃",
                "왜 지켜보나": "같은 강종이 2 LOT뿐이라 어느 쪽이 다른지 정할 수 없음"} for g in LEVEL_PAIR]
watch_rows += [{"구분": "설명됨", "LOT": k, "강종": lot_meta.loc[k, "GRADE"], "공정·센서": "세정 입구 CLN-IN",
                "무엇": "10/18 07:10 1분 · 최대 |Z| 4.0", "왜 지켜보나": "세정 입구가 평소처럼 오르내리는 중 가장 가파른 1분 (다른 LOT도 |Z| 3.5~3.8로 같은 모양)"} for k in EXPLAINED_SHORT]
table(pd.DataFrame(watch_rows))


### 다음에 확인할 기록

1. **CP-Z4 센서·전송·계산 기록:** 위 최종 선별 그림의 CP 시각. CP가 크게 뛰는 동안 같은 구역 온도는 거의 그대로였으므로 측정·기록 쪽을 먼저 본다.
2. **과시효 Z7·Z8 설정값·열전대 교정 기록:** "확실히 볼 것 (LOT 전체)" LOT과 같은 강종 다른 LOT의 날짜를 함께 놓고, 설정 온도가 달랐는지 확인한다. 통판 속도·두께 자료가 있으면 함께 맞춘다.
3. **지켜볼 것:** 같은 모양이 다른 LOT에서 반복되는지 새 자료가 들어올 때 다시 센다. Z7·Z8 2 LOT 차이는 같은 강종 LOT이 더 생기면 어느 쪽이 다른지 정할 수 있다.

---

## 부록

아래는 본문 결론을 추적하기 위한 상세 기록이다. 최종 확인 대상은 [위 최종 선별](#final-focus)에 모았다.

<a id="interval-evidence"></a>

### 부록 A. 탐지 구간과 실제 기록의 연결

In [ ]:
# 알려진 팀 해석은 240058 종료와 240061 초기만 별도로 표시한다. 다른 수리 전후는 시간·형태 관계다.
TEAM_CONFIRMED={240058:'종료 전환',240061:'초기 승온'}
reason_rows=[]
for e in intervals.itertuples():
    k=int(e.LOT);a,b=int(e.START_MIN),int(e.END_MIN);shape=states[k].loc[a:max(a,b-1)]
    reasons=[]
    for state in ['초기 승온','종료 전환']:
        if shape.eq(state).any():reasons.append(state+' · 팀 해석 연결' if TEAM_CONFIRMED.get(k)==state else state+' · 사후 형태')
    work=cal_work.loc[cal_work.STRT_DT.lt(e.END)&cal_work.END_DT.gt(e.START)]
    detect_near=cal_work.loc[cal_work.DETC_DT.between(e.START-pd.Timedelta(minutes=30),e.END+pd.Timedelta(minutes=30))]
    if len(work):reasons.append('실제 작업 교집합')
    if len(detect_near):reasons.append('감지 ±30분')
    if a<30:reasons.append('강종변경 직후 30분')
    if k in OVERAGE_LOTS and set(e.SENSORS.split(' · '))&{'TC-Z7','TC-Z8'}:reasons.append('과시효 LOT 수준 차이')
    if 'CLN-IN' in e.SENSORS:reasons.append('세정 입구 수준 대조 필요')
    level='팀 해석 연결' if any('팀 해석 연결' in x for x in reasons) else '시간·형태 관계' if reasons else '연결 기록 없음'
    reason_rows.append({'근거 수준':level,'겹친 기록 전체':' / '.join(reasons) or '-','작업 WO':' / '.join(work.WO_NO),'근접 감지 WO':' / '.join(detect_near.WO_NO),'원본 위치':f"[LOT {k}](05_all_data_timeseries_jinny_02.ipynb#lot-{k})"})
classified=pd.concat([intervals.reset_index(drop=True),pd.DataFrame(reason_rows)],axis=1)
LEVELS=['팀 해석 연결','시간·형태 관계','연결 기록 없음']
level_colors={'팀 해석 연결':'#8d99a6','시간·형태 관계':'#56b4e9','연결 기록 없음':'#17344d'}
methods_order=['조건별 범위 기준선','이동 Z-score','IsolationForest','CP 규칙']
cnt=classified.groupby(['METHOD','근거 수준']).size().unstack(fill_value=0).reindex(index=methods_order,columns=LEVELS,fill_value=0)
cnt["전체"]=cnt.sum(axis=1)
table(cnt)
none=classified.loc[classified['근거 수준'].eq('연결 기록 없음')]
note(f"**결과:** 전체 {len(classified)}개 탐지 구간 중 팀 해석으로 설명되는 구간 {int(cnt['팀 해석 연결'].sum())}개, 시간·형태가 기록과 겹치는 구간 {int(cnt['시간·형태 관계'].sum())}개, 겹치는 기록이 없는 구간 {len(none)}개다. "
     f"겹치는 기록이 없는 구간은 {', '.join(f'{k}({n}개)' for k, n in none.LOT.value_counts().head(4).items())} 순으로 많다. 아래 표의 '원본 위치'를 누르면 02에서 해당 LOT 그림으로 바로 간다.")
table(classified.sort_values(['LOT','START']))


### 근거를 읽는 기준

240058 종료와 240061 초기는 기존 팀 해석을 별도로 표시한다. 다른 수리 전후는 사후 형태와 시간 관계로 구분한다. 감지 ±30분은 검토 창이며 감지 원인이 아니다. 연결 기록 없음도 고장 확정이 아니다. 복수 근거를 모두 보존하고 대표 근거만으로 원인을 결정하지 않는다.

<a id="analysis-scope"></a>

### 부록 B. 분석 가능한 구간과 보류할 해석

In [ ]:
analysis_range=pd.DataFrame([
 ['측정 기록 공백','그 시각의 모든 센서 기록 없음','공백 표시·선 끊기·관측값 보존','공백 중의 급변·고장·정비 효과'],
 ['개별 센서 NaN','시각은 있으나 해당 센서 값 없음','해당 센서·분의 품질 표시·다른 센서 보존','빈 OP를 0%로 해석'],
 ['동일 강종 차이','유효 대조 LOT 수를 함께 확인','대조가 충분한 지속 차이의 원본 확인','같은 강종만으로 동일 운전 조건 확정'],
 ['수리 전후','전후 강종·관측 공백·구역별 변화 확인','원본·중앙값·IQR·변화율과 실제 교체 대조','전후 차이만으로 정비 효과 확정'],
 ['CP 변화','무차원 · 자체 참고 대비 이탈','방향·크기·지속·TC/OP 동시 변화 확인','CP로 강종·고장·정비 원인 확정'],
 ['감지·교체','감지 기준·정확한 교체 시각 미제공','실제 전후 관측과 이력 원문 대조','시간 순서만으로 감지 원인·교체 효과 확정']
],columns=['자료/구간','확인 조건','이번 분석에서 하는 것','현재 말할 수 없는 것'])
table(analysis_range)

### 학습 구간을 결정하는 표인가?

이 표는 이번 통계 비교에서 가능한 분석과 보류할 해석을 정리한다. 원본 삭제·예측 학습/제외 구간·실제 고장 라벨을 결정하지 않는다. 향후 모델링은 별도의 목표와 실제 운전·고장 정보를 확보해 진행해야 한다.

<a id="all-intervals"></a>

### 부록 C. 전수 결과와 상세 원본 연결

In [ ]:
table(review_catalogue)
table(classified[['METHOD','LOT','GRADE','START','END','SENSORS','근거 수준','겹친 기록 전체','원본 위치']])
table(pd.DataFrame(figure_checks))

note("[최종 선별로 돌아가기](#final-focus)")